<a href="https://colab.research.google.com/github/nmemranhussain/DiaVigil/blob/main/notebooks/00_DiaVigil_Reproducibility_Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Step 1: Cohort reconciliation**

In [1]:

from google.colab import drive
from pathlib import Path
import pandas as pd
import json

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/DiaVigil")
PROCESSED = BASE / "data" / "processed"

# Inspect saved cohort artifacts
for path in sorted(PROCESSED.glob("*")):
    if path.is_file():
        print(f"{path.name}: {path.stat().st_size:,} bytes")

# Check cohort counts if these files exist
for name in ["dev_cohort.parquet", "audit_test_cohort.parquet"]:
    path = PROCESSED / name
    if path.exists():
        df = pd.read_parquet(path)
        print(f"\n{name}")
        print("Rows:", len(df))
        print("Unique patients:",
              df["patient_nbr"].nunique()
              if "patient_nbr" in df else "Missing patient_nbr")
        print("Target prevalence:",
              df["target"].mean()
              if "target" in df else "Missing target")


Mounted at /content/drive
agent_audit_demonstration.joblib: 1,318 bytes
audit_test_cohort.parquet: 566,805 bytes
dev_cohort.parquet: 2,192,155 bytes
fairness_audit_results.joblib: 481,610 bytes
fairness_audit_results_reproduced_v2.joblib: 477,526 bytes
fairness_gap_patient_bootstrap_v1.csv: 376 bytes
feature_engineering_pipeline.joblib: 2,346 bytes
holdout_calibration_v2.csv: 731 bytes
model_calibration_artifacts.joblib: 468,095 bytes
model_calibration_artifacts_step20_backup.joblib: 468,095 bytes
model_calibration_artifacts_step23.joblib: 467,323 bytes
simulation_artifacts.joblib: 3,007 bytes
step23_holdout_evaluation.joblib: 477,533 bytes
step24_holdout_fairness_calibration.joblib: 5,437 bytes
step25_baseline_comparison.csv: 396 bytes
step26_paired_cluster_bootstrap.csv: 804 bytes
step27_bootstrap_summary.csv: 764 bytes
step27_model_comparison.csv: 389 bytes
step27_research_audit_report.md: 3,555 bytes
step29_final_research_decision_report.md: 2,028 bytes
step30_reproducibility_manif

**Step 2: Patient isolation and cohort integrity**

In [2]:

# STEP 2 — Independent cohort integrity checks
import pandas as pd
import numpy as np

dev = pd.read_parquet(PROCESSED / "dev_cohort.parquet")
holdout = pd.read_parquet(PROCESSED / "audit_test_cohort.parquet")

required = {"patient_nbr", "encounter_id", "target"}
for name, df in [("Development", dev), ("Holdout", holdout)]:
    missing_cols = required - set(df.columns)
    print(f"\n{name} cohort")
    print("Missing required columns:", missing_cols)
    assert not missing_cols, f"{name} cohort lacks required columns"

    print("Missing patient IDs:", df["patient_nbr"].isna().sum())
    print("Missing encounter IDs:", df["encounter_id"].isna().sum())
    print("Duplicate encounter IDs:", df["encounter_id"].duplicated().sum())
    print("Missing targets:", df["target"].isna().sum())
    print("Target values:", sorted(df["target"].dropna().unique()))

    assert df["patient_nbr"].notna().all()
    assert df["encounter_id"].notna().all()
    assert df["target"].notna().all()
    assert set(df["target"].unique()).issubset({0, 1})
    assert not df["encounter_id"].duplicated().any()

dev_patients = set(dev["patient_nbr"])
test_patients = set(holdout["patient_nbr"])
dev_encounters = set(dev["encounter_id"])
test_encounters = set(holdout["encounter_id"])

print("\nCROSS-PARTITION CHECKS")
print("Patients in both:", len(dev_patients & test_patients))
print("Encounters in both:", len(dev_encounters & test_encounters))
print("Combined encounters:", len(dev) + len(holdout))
print("Combined unique patients:",
      len(dev_patients | test_patients))

assert not (dev_patients & test_patients), "PATIENT LEAKAGE DETECTED"
assert not (dev_encounters & test_encounters), "ENCOUNTER OVERLAP DETECTED"

print("\nPASS: partition integrity checks completed.")



Development cohort
Missing required columns: set()
Missing patient IDs: 0
Missing encounter IDs: 0
Duplicate encounter IDs: 0
Missing targets: 0
Target values: [np.int64(0), np.int64(1)]

Holdout cohort
Missing required columns: set()
Missing patient IDs: 0
Missing encounter IDs: 0
Duplicate encounter IDs: 0
Missing targets: 0
Target values: [np.int64(0), np.int64(1)]

CROSS-PARTITION CHECKS
Patients in both: 0
Encounters in both: 0
Combined encounters: 100114
Combined unique patients: 70439

PASS: partition integrity checks completed.


**Step 3: Inspect the saved model artifacts**

In [3]:
!pip install catboost -q

# STEP 3 — Inspect artifact metadata safely
import joblib
import sklearn
import catboost
import sys

artifact_names = [
    "model_calibration_artifacts.joblib",
    "feature_engineering_pipeline.joblib",
    "fairness_audit_results.joblib",
    "simulation_artifacts.joblib",
]

print("Python:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("CatBoost:", catboost.__version__)

for name in artifact_names:
    path = PROCESSED / name
    if not path.exists():
        print(f"\nMissing: {name}")
        continue

    print(f"\n--- {name} ---")
    try:
        obj = joblib.load(path)
        print("Object type:", type(obj))

        if isinstance(obj, dict):
            print("Keys:", list(obj.keys()))
            for key, value in obj.items():
                if isinstance(value, (str, int, float, bool, type(None))):
                    print(f"  {key}: {value}")
                else:
                    print(f"  {key}: {type(value).__name__}")
        elif hasattr(obj, "get_params"):
            print("Parameters:", obj.get_params())
        elif hasattr(obj, "named_steps"):
            print("Pipeline steps:", list(obj.named_steps.keys()))
    except Exception as e:
        print("LOAD ERROR:", repr(e))

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
scikit-learn: 1.6.1
CatBoost: 1.2.10

--- model_calibration_artifacts.joblib ---
LOAD ERROR: AttributeError("Can't get attribute 'ClinicalDomainFeatureEngineer' on <module '__main__'>")

--- feature_engineering_pipeline.joblib ---
LOAD ERROR: AttributeError("Can't get attribute 'ClinicalDomainFeatureEngineer' on <module '__main__'>")

--- fairness_audit_results.joblib ---
Object type: <class 'dict'>
Keys: ['age_audit', 'gender_audit', 'race_audit', 'test_cal_probs', 'test_preds', 'y_test']
  age_audit: DataFrame
  gender_audit: DataFrame
  race_audit: DataFrame
  test_cal_probs: ndarray
  test_preds: ndarray
  y_test: ndarray

--- simulation_artifacts.joblib ---
Object type: <class 'dict'>
Keys: ['allocation_summary', 'capacity_frontier', 'monte_carlo_bounds']
  allocation_summary: DataFrame
  capacity_frontier: DataFrame
  monte_carlo_bounds: dict


**Step 4 — Recompute the holdout metrics**

In [4]:

# STEP 4 — Independent recomputation from saved holdout predictions

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
)

artifact_path = PROCESSED / "fairness_audit_results.joblib"
audit = joblib.load(artifact_path)

y = np.asarray(audit["y_test"]).astype(int)
p = np.asarray(audit["test_cal_probs"], dtype=float)
saved_preds = np.asarray(audit["test_preds"]).astype(int)

assert len(y) == len(p) == len(saved_preds)
assert np.isfinite(p).all()
assert ((p >= 0) & (p <= 1)).all()

# Verify predictions at the published threshold
threshold = 0.162
recomputed_preds = (p >= threshold).astype(int)

print("Predictions match threshold 0.162:",
      np.array_equal(saved_preds, recomputed_preds))

print("\nHOLDOUT METRICS")
print("N:", len(y))
print("Positive outcomes:", int(y.sum()))
print("Prevalence:", round(y.mean(), 6))
print("PR-AUC:", round(average_precision_score(y, p), 6))
print("ROC-AUC:", round(roc_auc_score(y, p), 6))
print("Brier score:", round(brier_score_loss(y, p), 6))

# Ten equal-width ECE bins
edges = np.linspace(0, 1, 11)
bin_ids = np.minimum(np.digitize(p, edges[1:-1]), 9)
ece = 0.0
rows = []

for b in range(10):
    mask = bin_ids == b
    if not mask.any():
        continue
    mean_p = p[mask].mean()
    observed = y[mask].mean()
    n = int(mask.sum())
    ece += n / len(y) * abs(mean_p - observed)
    rows.append({
        "bin": b + 1,
        "n": n,
        "mean_predicted_risk": mean_p,
        "observed_rate": observed,
        "absolute_gap": abs(mean_p - observed),
    })

print("ECE (10 equal-width bins):", round(ece, 6))
display(pd.DataFrame(rows))

tn, fp, fn, tp = confusion_matrix(
    y, recomputed_preds, labels=[0, 1]
).ravel()

print("\nCONFUSION MATRIX COUNTS")
print({"TN": tn, "FP": fp, "FN": fn, "TP": tp})
print("Sensitivity:", round(tp / (tp + fn), 6))
print("Precision:", round(tp / (tp + fp), 6))
print("False-negative rate:", round(fn / (fn + tp), 6))


Predictions match threshold 0.162: True

HOLDOUT METRICS
N: 19875
Positive outcomes: 2240
Prevalence: 0.112704
PR-AUC: 0.203595
ROC-AUC: 0.664123
Brier score: 0.096154
ECE (10 equal-width bins): 0.002629


,bin,n,mean_predicted_risk,observed_rate,absolute_gap
0,1,9645,0.066373,0.065630,0.000743
1,2,8717,0.136016,0.136515,0.000499
2,3,928,0.235979,0.246767,0.010788
3,4,423,0.327025,0.297872,0.029152
4,5,100,0.469775,0.400000,0.069775
5,6,60,0.521241,0.333333,0.187908
6,9,1,0.857143,1.000000,0.142857
7,10,1,1.000000,1.000000,0.000000



CONFUSION MATRIX COUNTS
{'TN': np.int64(15546), 'FP': np.int64(2089), 'FN': np.int64(1587), 'TP': np.int64(653)}
Sensitivity: 0.291518
Precision: 0.238147
False-negative rate: 0.708482


**Step 5 — Recompute the fairness gap from the actual holdout rows**

In [5]:

# STEP 5 — Recompute subgroup FNRs from the holdout cohort

holdout = pd.read_parquet(
    PROCESSED / "audit_test_cohort.parquet"
).copy()

# Confirm row ordering before joining predictions.
assert len(holdout) == len(y)

# First inspect available columns.
print("Demographic columns available:",
      [c for c in ["age", "gender", "race"] if c in holdout.columns])

# Attach saved predictions in the same order only after checking
# that the artifact and holdout rows correspond.
# If the artifact has no encounter IDs, row order must be verified
# from the original notebook before treating this join as definitive.
holdout["audit_y"] = y
holdout["audit_p"] = p
holdout["audit_pred"] = recomputed_preds

def calculate_fnr_by_group(df, group_col):
    results = []
    for group, part in df.groupby(group_col, dropna=False, observed=True):
        positives = int((part["audit_y"] == 1).sum())
        fn = int(((part["audit_y"] == 1) &
                  (part["audit_pred"] == 0)).sum())
        tp = int(((part["audit_y"] == 1) &
                  (part["audit_pred"] == 1)).sum())

        results.append({
            "group": str(group),
            "n_encounters": len(part),
            "positive_outcomes": positives,
            "FN": fn,
            "TP": tp,
            "FNR": fn / positives if positives else np.nan,
        })

    result = pd.DataFrame(results)
    valid = result.dropna(subset=["FNR"])

    print(f"\n{group_col.upper()} AUDIT")
    display(result)

    if len(valid):
        gap = valid["FNR"].max() - valid["FNR"].min()
        print("Maximum FNR gap:", round(gap, 6))

for col in ["age", "gender", "race"]:
    if col in holdout.columns:
        calculate_fnr_by_group(holdout, col)


Demographic columns available: ['age', 'gender', 'race']

AGE AUDIT


,group,n_encounters,positive_outcomes,FN,TP,FNR
0,[0-10),29,1,1,0,1.000000
1,[10-20),128,11,10,1,0.909091
2,[20-30),307,47,31,16,0.659574
3,[30-40),767,89,44,45,0.494382
4,[40-50),1893,191,125,66,0.654450
5,[50-60),3431,341,224,117,0.656891
6,[60-70),4433,486,357,129,0.734568
7,[70-80),5041,603,430,173,0.713101
8,[80-90),3328,413,315,98,0.762712
9,[90-100),518,58,50,8,0.862069


Maximum FNR gap: 0.505618

GENDER AUDIT


,group,n_encounters,positive_outcomes,FN,TP,FNR
0,Female,10792,1188,837,351,0.704545
1,Male,9083,1052,750,302,0.712928


Maximum FNR gap: 0.008382

RACE AUDIT


,group,n_encounters,positive_outcomes,FN,TP,FNR
0,?,418,24,22,2,0.916667
1,AfricanAmerican,3787,417,291,126,0.697842
2,Asian,145,13,8,5,0.615385
3,Caucasian,14858,1731,1235,496,0.713460
4,Hispanic,376,33,18,15,0.545455
5,Other,291,22,13,9,0.590909


Maximum FNR gap: 0.371212


**Step 6 — Verify row alignment and locate the original class**

In [6]:

# STEP 6 — Trace prediction alignment and custom class definition

import json

notebook_files = list(BASE.rglob("*.ipynb"))
print("Notebook files found:", len(notebook_files))

terms = [
    "ClinicalDomainFeatureEngineer",
    "test_cal_probs",
    "y_test",
    "audit_test_cohort",
]

for path in notebook_files:
    try:
        nb = json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        print("Could not read:", path.name, repr(e))
        continue

    matches = []
    for i, cell in enumerate(nb.get("cells", [])):
        source = "".join(cell.get("source", []))
        if any(term in source for term in terms):
            matches.append((i, source))

    if matches:
        print(f"\n{'='*70}\nNOTEBOOK: {path}")
        for i, source in matches:
            print(f"\n--- Cell {i} ---\n")
            print(source[:5000])


Notebook files found: 0


In [7]:

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

all_notebooks = list(DRIVE_ROOT.rglob("*.ipynb"))

print("Total notebooks found in Google Drive:", len(all_notebooks))

for path in all_notebooks:
    if "diavigil" in str(path).lower():
        print(path)


Total notebooks found in Google Drive: 16
/content/drive/MyDrive/Colab Notebooks/DiaVigil_v2.ipynb
/content/drive/MyDrive/Colab Notebooks/02_diavigil_clinical_feature_engineering_pipeline.ipynb
/content/drive/MyDrive/Colab Notebooks/03_diavigil_nested_cv_and_probability_calibration.ipynb
/content/drive/MyDrive/Colab Notebooks/04_diavigil_algorithmic_fairness_and_subgroup_audit (1).ipynb
/content/drive/MyDrive/Colab Notebooks/04_diavigil_algorithmic_fairness_and_subgroup_audit.ipynb
/content/drive/MyDrive/Colab Notebooks/05_diavigil_agentic_triage_and_faithfulness_critic.ipynb
/content/drive/MyDrive/Colab Notebooks/06_diavigil_dynamic_resource_allocation_simulation (1).ipynb
/content/drive/MyDrive/Colab Notebooks/06_diavigil_dynamic_resource_allocation_simulation.ipynb
/content/drive/MyDrive/Colab Notebooks/00_DiaVigil_Reproducibility_Audit.ipynb


**Step 7 — Restore the custom class in Colab**

In [8]:

# STEP 7 — Extract the exact custom class source
import json
from pathlib import Path

NOTEBOOK_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks"
)

notebook_path = (
    NOTEBOOK_DIR /
    "03_diavigil_nested_cv_and_probability_calibration.ipynb"
)

assert notebook_path.exists(), (
    f"Original notebook not found: {notebook_path}"
)

nb = json.loads(notebook_path.read_text(encoding="utf-8"))

matches = []

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    if "class ClinicalDomainFeatureEngineer" in source:
        matches.append((i, source))

print("Matching cells:", len(matches))

for i, source in matches:
    print(f"\n--- Original Notebook 03, cell {i} ---")
    print(source)


Matching cells: 0


**Step 8 — Load the saved artifacts**

In [9]:

# STEP 8 — Load saved artifacts without modifying them
import joblib
import sklearn
import catboost
from pathlib import Path

print("scikit-learn:", sklearn.__version__)
print("CatBoost:", catboost.__version__)

artifact_dir = Path(
    "/content/drive/MyDrive/DiaVigil/data/processed"
)

artifact_path = (
    artifact_dir / "model_calibration_artifacts.joblib"
)
pipeline_path = (
    artifact_dir / "feature_engineering_pipeline.joblib"
)

for path in [artifact_path, pipeline_path]:
    print(f"\nLoading: {path.name}")
    assert path.exists(), f"Missing artifact: {path}"

    try:
        obj = joblib.load(path)
        print("Loaded successfully:", type(obj))

        if isinstance(obj, dict):
            print("Keys:", list(obj.keys()))
            for key, value in obj.items():
                print(f"  {key}: {type(value).__name__}")

        elif hasattr(obj, "named_steps"):
            print("Pipeline steps:", list(obj.named_steps.keys()))

    except Exception as e:
        print("LOAD FAILED:", repr(e))


scikit-learn: 1.6.1
CatBoost: 1.2.10

Loading: model_calibration_artifacts.joblib
LOAD FAILED: AttributeError("Can't get attribute 'ClinicalDomainFeatureEngineer' on <module '__main__'>")

Loading: feature_engineering_pipeline.joblib
LOAD FAILED: AttributeError("Can't get attribute 'ClinicalDomainFeatureEngineer' on <module '__main__'>")


**Step 9 — Find the exact class definition across all original notebooks**

In [10]:

# STEP 9 — Locate the original class definition

import json
from pathlib import Path

NOTEBOOK_DIR = Path("/content/drive/MyDrive/Colab Notebooks")

terms = [
    "ClinicalDomainFeatureEngineer",
    "def extract_elixhauser_flags",
    "ELIXHAUSER_ICD9_RULES",
]

matches = 0

for path in sorted(NOTEBOOK_DIR.glob("*.ipynb")):
    try:
        nb = json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        print("Could not read:", path.name, repr(e))
        continue

    for i, cell in enumerate(nb.get("cells", [])):
        source = "".join(cell.get("source", []))
        if "ClinicalDomainFeatureEngineer" in source:
            matches += 1
            print("\n" + "=" * 80)
            print("FILE:", path.name)
            print("CELL:", i)
            print("-" * 80)
            print(source)

print("\nTotal matching cells:", matches)



FILE: 00_DiaVigil_Reproducibility_Audit.ipynb
CELL: 11
--------------------------------------------------------------------------------

# STEP 6 — Trace prediction alignment and custom class definition

import json

notebook_files = list(BASE.rglob("*.ipynb"))
print("Notebook files found:", len(notebook_files))

terms = [
    "ClinicalDomainFeatureEngineer",
    "test_cal_probs",
    "y_test",
    "audit_test_cohort",
]

for path in notebook_files:
    try:
        nb = json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        print("Could not read:", path.name, repr(e))
        continue

    matches = []
    for i, cell in enumerate(nb.get("cells", [])):
        source = "".join(cell.get("source", []))
        if any(term in source for term in terms):
            matches.append((i, source))

    if matches:
        print(f"\n{'='*70}\nNOTEBOOK: {path}")
        for i, source in matches:
            print(f"\n--- Cell {i} ---\n")
            print(source[:5000])

**Step 10 — Restore the class and load the artifacts**

In [11]:

# STEP 10 — Restore original class definitions safely

import json
import re
import joblib
import pandas as pd
from pathlib import Path
from sklearn.base import BaseEstimator, TransformerMixin

NOTEBOOK_DIR = Path("/content/drive/MyDrive/Colab Notebooks")
SOURCE_NOTEBOOK = (
    NOTEBOOK_DIR /
    "05_diavigil_agentic_triage_and_faithfulness_critic.ipynb"
)

nb = json.loads(SOURCE_NOTEBOOK.read_text(encoding="utf-8"))

candidates = []
for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    if (
        "class ClinicalDomainFeatureEngineer" in source
        and "def extract_elixhauser_flags" in source
        and "ELIXHAUSER_ICD9_RULES" in source
    ):
        candidates.append((i, source))

assert candidates, "Could not find the complete original class cell."

# Use the original definition cell, but do not execute its artifact-loading
# statements. Extract only the prefix ending immediately before that section.
cell_index, source = candidates[0]
marker = "# Load pipeline, champion model, calibrator, and audit test set"

assert marker in source, "Expected boundary marker was not found."

definition_source = source.split(marker, 1)[0]

# Remove any IPython shell execution lines (like !pip install) so exec() can compile successfully
clean_source_lines = [line for line in definition_source.splitlines() if not line.strip().startswith("!")]
clean_definition_source = "\n".join(clean_source_lines)

exec(compile(clean_definition_source, str(SOURCE_NOTEBOOK), "exec"), globals())

assert "ClinicalDomainFeatureEngineer" in globals()
print("Restored class from Notebook 05, cell", cell_index)
print("Class:", ClinicalDomainFeatureEngineer)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Restored class from Notebook 05, cell 2
Class: <class '__main__.ClinicalDomainFeatureEngineer'>


**Step 11 — Retry loading the saved artifacts**

In [12]:

import joblib
from pathlib import Path

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/DiaVigil/data/processed"
)

loaded_artifacts = {}

for filename in [
    "feature_engineering_pipeline.joblib",
    "model_calibration_artifacts.joblib",
]:
    path = ARTIFACT_DIR / filename
    print(f"\n--- Loading {filename} ---")

    try:
        obj = joblib.load(path)
        loaded_artifacts[filename] = obj
        print("SUCCESS:", type(obj))

        if isinstance(obj, dict):
            print("Keys:", list(obj.keys()))
            for key, value in obj.items():
                print(f"  {key}: {type(value).__name__}")

        elif hasattr(obj, "named_steps"):
            print("Pipeline steps:", list(obj.named_steps.keys()))

    except Exception as exc:
        print("FAILED:", type(exc).__name__, str(exc))

print("\nSuccessfully loaded:",
      list(loaded_artifacts.keys()))



--- Loading feature_engineering_pipeline.joblib ---
SUCCESS: <class 'sklearn.pipeline.Pipeline'>
Pipeline steps: ['domain_engineering', 'preprocessor']

--- Loading model_calibration_artifacts.joblib ---
SUCCESS: <class 'dict'>
Keys: ['fitted_pipeline', 'champion_model', 'calibrator', 'optimal_threshold']
  fitted_pipeline: Pipeline
  champion_model: CatBoostClassifier
  calibrator: CalibratedClassifierCV
  optimal_threshold: float64

Successfully loaded: ['feature_engineering_pipeline.joblib', 'model_calibration_artifacts.joblib']


**Step 12 — Inspect the loaded model and calibration metadata**

In [13]:

# STEP 12 — Inspect model, calibrator, pipeline, and threshold

import numpy as np
import sklearn
import catboost

artifacts = loaded_artifacts["model_calibration_artifacts.joblib"]
pipeline = artifacts["fitted_pipeline"]
model = artifacts["champion_model"]
calibrator = artifacts["calibrator"]
threshold = float(artifacts["optimal_threshold"])

print("scikit-learn:", sklearn.__version__)
print("CatBoost:", catboost.__version__)

print("\nSAVED THRESHOLD:", threshold)

print("\nPIPELINE STEPS:")
for name, step in pipeline.named_steps.items():
    print(name, "->", type(step).__name__)

print("\nCHAMPION MODEL:")
print("Type:", type(model).__name__)
print("Parameters:", model.get_params())

print("\nCALIBRATOR:")
print("Type:", type(calibrator).__name__)
print("Method:", getattr(calibrator, "method", "not exposed"))
print("CV setting:", getattr(calibrator, "cv", "not exposed"))

if hasattr(calibrator, "calibrated_classifiers_"):
    print("Fitted calibrated classifiers:",
          len(calibrator.calibrated_classifiers_))

    for i, fitted in enumerate(calibrator.calibrated_classifiers_):
        print(f"Calibrated estimator {i}:",
              type(getattr(fitted, "estimator", None)).__name__)
        print("Calibration mappings:",
              [type(c).__name__ for c in
               getattr(fitted, "calibrators", [])])

print("\nARTIFACT KEYS:", list(artifacts.keys()))


scikit-learn: 1.6.1
CatBoost: 1.2.10

SAVED THRESHOLD: 0.162020202020202

PIPELINE STEPS:
domain_engineering -> ClinicalDomainFeatureEngineer
preprocessor -> ColumnTransformer

CHAMPION MODEL:
Type: CatBoostClassifier
Parameters: {'iterations': 400, 'learning_rate': 0.05, 'depth': 6, 'random_seed': 42, 'verbose': 100, 'auto_class_weights': 'Balanced'}

CALIBRATOR:
Type: CalibratedClassifierCV
Method: isotonic
CV setting: prefit
Fitted calibrated classifiers: 1
Calibrated estimator 0: CatBoostClassifier
Calibration mappings: ['IsotonicRegression']

ARTIFACT KEYS: ['fitted_pipeline', 'champion_model', 'calibrator', 'optimal_threshold']


**Step 13 — Verify holdout prediction alignment**

In [14]:

# STEP 13 — Check holdout array dimensions and saved threshold

import joblib
import pandas as pd
import numpy as np

holdout = pd.read_parquet(
    ARTIFACT_DIR / "audit_test_cohort.parquet"
).reset_index(drop=True)

audit = joblib.load(
    ARTIFACT_DIR / "fairness_audit_results.joblib"
)

y_saved = np.asarray(audit["y_test"]).astype(int)
p_saved = np.asarray(audit["test_cal_probs"], dtype=float)
pred_saved = np.asarray(audit["test_preds"]).astype(int)

print("Holdout rows:", len(holdout))
print("Saved labels:", len(y_saved))
print("Saved probabilities:", len(p_saved))
print("Saved predictions:", len(pred_saved))

print("Saved labels match holdout target in current row order:",
      np.array_equal(holdout["target"].to_numpy(), y_saved))

print("Saved predictions match saved threshold:",
      np.array_equal(
          pred_saved,
          (p_saved >= threshold).astype(int)
      ))

assert len(holdout) == len(y_saved) == len(p_saved) == len(pred_saved)


Holdout rows: 19875
Saved labels: 19875
Saved probabilities: 19875
Saved predictions: 19875
Saved labels match holdout target in current row order: True
Saved predictions match saved threshold: True


**Step 14 — Regenerate the holdout predictions independently**

In [15]:

# STEP 14 — Reproduce holdout probabilities from saved artifacts

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

artifacts = loaded_artifacts["model_calibration_artifacts.joblib"]
pipeline = artifacts["fitted_pipeline"]
calibrator = artifacts["calibrator"]
threshold = float(artifacts["optimal_threshold"])

holdout = pd.read_parquet(
    ARTIFACT_DIR / "audit_test_cohort.parquet"
).reset_index(drop=True)

drop_cols = ["target", "encounter_id", "patient_nbr"]
X_holdout = holdout.drop(columns=drop_cols, errors="ignore")
y_holdout = holdout["target"].to_numpy(dtype=int)

# Transform using the saved fitted pipeline; do not call fit or fit_transform.
X_transformed = pipeline.transform(X_holdout)

# Regenerate calibrated probabilities
p_reproduced = calibrator.predict_proba(X_transformed)[:, 1]

audit = joblib.load(
    ARTIFACT_DIR / "fairness_audit_results.joblib"
)
p_saved = np.asarray(audit["test_cal_probs"], dtype=float)
y_saved = np.asarray(audit["y_test"], dtype=int)

print("Transformed feature shape:", X_transformed.shape)
print("Probability array lengths:", len(p_reproduced), len(p_saved))
print("Labels match:", np.array_equal(y_holdout, y_saved))

if len(p_reproduced) == len(p_saved):
    diff = np.abs(p_reproduced - p_saved)
    print("Maximum absolute probability difference:", diff.max())
    print("Mean absolute probability difference:", diff.mean())
    print("Probabilities match within 1e-8:",
          np.allclose(p_reproduced, p_saved, atol=1e-8, rtol=0))

print("\nREPRODUCED HOLDOUT METRICS")
print("PR-AUC:", average_precision_score(y_holdout, p_reproduced))
print("ROC-AUC:", roc_auc_score(y_holdout, p_reproduced))
print("Brier score:", brier_score_loss(y_holdout, p_reproduced))
print("Threshold:", threshold)
print("Predicted positive count:",
      int((p_reproduced >= threshold).sum()))


Transformed feature shape: (19875, 98)
Probability array lengths: 19875 19875
Labels match: True
Maximum absolute probability difference: 0.0
Mean absolute probability difference: 0.0
Probabilities match within 1e-8: True

REPRODUCED HOLDOUT METRICS
PR-AUC: 0.20359523855414996
ROC-AUC: 0.6641228001336626
Brier score: 0.09615431105958255
Threshold: 0.162020202020202
Predicted positive count: 2742


**Step 15 — Recompute fairness and calibration on the reproduced predictions**

In [16]:

# STEP 15 — Independent subgroup fairness and calibration audit

import numpy as np
import pandas as pd
from scipy.stats import beta

# Use the holdout and probabilities reproduced in Step 14
df = holdout.copy()
df["audit_y"] = y_holdout
df["audit_p"] = p_reproduced
df["audit_pred"] = (p_reproduced >= threshold).astype(int)

assert len(df) == len(p_reproduced)
assert np.array_equal(df["target"].to_numpy(), y_holdout)

def exact_binomial_ci(events, total, alpha=0.05):
    """Clopper-Pearson interval for a binomial proportion."""
    if total == 0:
        return (np.nan, np.nan)

    lower = 0.0 if events == 0 else beta.ppf(
        alpha / 2, events, total - events + 1
    )
    upper = 1.0 if events == total else beta.ppf(
        1 - alpha / 2, events + 1, total - events
    )
    return float(lower), float(upper)

def subgroup_fnr_audit(data, group_col):
    rows = []

    for group, part in data.groupby(
        group_col, dropna=False, observed=True
    ):
        positives = int((part["audit_y"] == 1).sum())
        fn = int((
            (part["audit_y"] == 1) &
            (part["audit_pred"] == 0)
        ).sum())

        if positives:
            fnr = fn / positives
            lo, hi = exact_binomial_ci(fn, positives)
        else:
            fnr, lo, hi = np.nan, np.nan, np.nan

        rows.append({
            "group": str(group),
            "encounters": len(part),
            "positive_outcomes": positives,
            "false_negatives": fn,
            "FNR": fnr,
            "FNR_CI95_low": lo,
            "FNR_CI95_high": hi,
        })

    result = pd.DataFrame(rows)
    valid = result.dropna(subset=["FNR"])

    print(f"\n--- {group_col.upper()} FNR AUDIT ---")
    display(result)

    if len(valid) >= 2:
        high = valid.loc[valid["FNR"].idxmax()]
        low = valid.loc[valid["FNR"].idxmin()]
        print(
            "Observed max FNR gap:",
            round(high["FNR"] - low["FNR"], 6)
        )
        print("Highest FNR group:", high["group"])
        print("Lowest FNR group:", low["group"])

for col in ["age", "gender", "race"]:
    if col in df.columns:
        subgroup_fnr_audit(df, col)

# Calibration: 10 approximately equal-sized bins
cal = pd.DataFrame({
    "y": y_holdout,
    "p": p_reproduced
}).sort_values("p").reset_index(drop=True)

cal["bin"] = pd.qcut(
    cal.index, q=10, labels=False, duplicates="drop"
)

calibration_rows = []
ece = 0.0

for b, part in cal.groupby("bin", observed=True):
    n = len(part)
    mean_pred = part["p"].mean()
    observed = part["y"].mean()

    ece += (n / len(cal)) * abs(mean_pred - observed)

    calibration_rows.append({
        "bin": int(b) + 1,
        "n": n,
        "mean_predicted_risk": mean_pred,
        "observed_rate": observed,
        "absolute_gap": abs(mean_pred - observed),
    })

print("\n--- HOLDOUT CALIBRATION: 10 EQUAL-FREQUENCY BINS ---")
display(pd.DataFrame(calibration_rows))
print("Equal-frequency ECE:", round(ece, 6))



--- AGE FNR AUDIT ---


,group,encounters,positive_outcomes,false_negatives,FNR,FNR_CI95_low,FNR_CI95_high
0,[0-10),29,1,1,1.000000,0.025000,1.000000
1,[10-20),128,11,10,0.909091,0.587220,0.997701
2,[20-30),307,47,31,0.659574,0.506868,0.791359
3,[30-40),767,89,44,0.494382,0.386665,0.602484
4,[40-50),1893,191,125,0.654450,0.582393,0.721629
5,[50-60),3431,341,224,0.656891,0.603841,0.707190
6,[60-70),4433,486,357,0.734568,0.692930,0.773325
7,[70-80),5041,603,430,0.713101,0.675187,0.748913
8,[80-90),3328,413,315,0.762712,0.718671,0.802942
9,[90-100),518,58,50,0.862069,0.746190,0.938520


Observed max FNR gap: 0.505618
Highest FNR group: [0-10)
Lowest FNR group: [30-40)

--- GENDER FNR AUDIT ---


,group,encounters,positive_outcomes,false_negatives,FNR,FNR_CI95_low,FNR_CI95_high
0,Female,10792,1188,837,0.704545,0.677697,0.730378
1,Male,9083,1052,750,0.712928,0.684546,0.740113


Observed max FNR gap: 0.008382
Highest FNR group: Male
Lowest FNR group: Female

--- RACE FNR AUDIT ---


,group,encounters,positive_outcomes,false_negatives,FNR,FNR_CI95_low,FNR_CI95_high
0,?,418,24,22,0.916667,0.730027,0.989744
1,AfricanAmerican,3787,417,291,0.697842,0.651276,0.741574
2,Asian,145,13,8,0.615385,0.315778,0.861421
3,Caucasian,14858,1731,1235,0.713460,0.691522,0.734673
4,Hispanic,376,33,18,0.545455,0.363507,0.718934
5,Other,291,22,13,0.590909,0.363547,0.792907


Observed max FNR gap: 0.371212
Highest FNR group: ?
Lowest FNR group: Hispanic

--- HOLDOUT CALIBRATION: 10 EQUAL-FREQUENCY BINS ---


,bin,n,mean_predicted_risk,observed_rate,absolute_gap
0,1,1988,0.040810,0.040744,0.000065
1,2,1987,0.059162,0.065425,0.006263
2,3,1988,0.071206,0.056841,0.014365
3,4,1987,0.072099,0.082033,0.009935
4,5,1988,0.093683,0.088531,0.005152
5,6,1987,0.108435,0.106190,0.002245
6,7,1987,0.127479,0.118269,0.009211
7,8,1988,0.136017,0.139839,0.003822
8,9,1987,0.164742,0.167086,0.002344
9,10,1988,0.265088,0.262072,0.003016


Equal-frequency ECE: 0.005642


**Step 16 — Sensitivity analysis for fairness gaps**

In [17]:

# STEP 16 — Fairness sensitivity analysis
# Uses df created in Step 15.

import numpy as np
import pandas as pd

def summarize_fnr_gap(data, group_col, min_positives=1):
    rows = []

    for group, part in data.groupby(
        group_col, dropna=False, observed=True
    ):
        positives = int((part["audit_y"] == 1).sum())
        fn = int((
            (part["audit_y"] == 1) &
            (part["audit_pred"] == 0)
        ).sum())

        rows.append({
            "group": str(group),
            "encounters": len(part),
            "positive_outcomes": positives,
            "FN": fn,
            "FNR": fn / positives if positives else np.nan
        })

    result = pd.DataFrame(rows)
    print(f"\n{group_col}: all groups")
    display(result)

    eligible = result[
        (result["positive_outcomes"] >= min_positives) &
        result["FNR"].notna()
    ]

    if len(eligible) >= 2:
        gap = eligible["FNR"].max() - eligible["FNR"].min()
        print(
            f"Gap among groups with >= {min_positives} positives:",
            round(gap, 6)
        )
        print(
            "Groups defining gap:",
            eligible.loc[eligible["FNR"].idxmax(), "group"],
            "vs.",
            eligible.loc[eligible["FNR"].idxmin(), "group"]
        )

    return result

age_results = summarize_fnr_gap(df, "age", min_positives=50)
gender_results = summarize_fnr_gap(df, "gender", min_positives=50)
race_results = summarize_fnr_gap(df, "race", min_positives=50)

# Secondary race analysis excluding unknown/missing labels.
race_known = df[
    df["race"].notna() &
    (df["race"].astype(str).str.strip() != "?")
].copy()

print("\nRace sensitivity analysis excluding unknown labels")
race_known_results = summarize_fnr_gap(
    race_known, "race", min_positives=1
)



age: all groups


,group,encounters,positive_outcomes,FN,FNR
0,[0-10),29,1,1,1.000000
1,[10-20),128,11,10,0.909091
2,[20-30),307,47,31,0.659574
3,[30-40),767,89,44,0.494382
4,[40-50),1893,191,125,0.654450
5,[50-60),3431,341,224,0.656891
6,[60-70),4433,486,357,0.734568
7,[70-80),5041,603,430,0.713101
8,[80-90),3328,413,315,0.762712
9,[90-100),518,58,50,0.862069


Gap among groups with >= 50 positives: 0.367687
Groups defining gap: [90-100) vs. [30-40)

gender: all groups


,group,encounters,positive_outcomes,FN,FNR
0,Female,10792,1188,837,0.704545
1,Male,9083,1052,750,0.712928


Gap among groups with >= 50 positives: 0.008382
Groups defining gap: Male vs. Female

race: all groups


,group,encounters,positive_outcomes,FN,FNR
0,?,418,24,22,0.916667
1,AfricanAmerican,3787,417,291,0.697842
2,Asian,145,13,8,0.615385
3,Caucasian,14858,1731,1235,0.713460
4,Hispanic,376,33,18,0.545455
5,Other,291,22,13,0.590909


Gap among groups with >= 50 positives: 0.015619
Groups defining gap: Caucasian vs. AfricanAmerican

Race sensitivity analysis excluding unknown labels

race: all groups


,group,encounters,positive_outcomes,FN,FNR
0,AfricanAmerican,3787,417,291,0.697842
1,Asian,145,13,8,0.615385
2,Caucasian,14858,1731,1235,0.713460
3,Hispanic,376,33,18,0.545455
4,Other,291,22,13,0.590909


Gap among groups with >= 1 positives: 0.168006
Groups defining gap: Caucasian vs. Hispanic


**Step 17 — Audit the threshold-selection procedure**

In [18]:

# STEP 17 — Inspect calibration and threshold-selection source

import json
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "03_diavigil_nested_cv_and_probability_calibration.ipynb"
)

assert notebook_path.exists(), (
    f"Notebook not found: {notebook_path}"
)

nb = json.loads(notebook_path.read_text(encoding="utf-8"))

search_terms = [
    "CalibratedClassifierCV",
    "calibrator.fit",
    "cal_val_probs",
    "threshold_grid",
    "total_costs",
    "optimal_threshold",
    "C_fp",
    "C_fn",
]

for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))

    matched_terms = [
        term for term in search_terms if term in source
    ]

    if matched_terms:
        print("\n" + "=" * 75)
        print(f"CELL {i} | Matches: {matched_terms}")
        print("=" * 75)
        print(source[:12000])

        # Show existing outputs, if present
        outputs = cell.get("outputs", [])
        for output in outputs:
            text_output = output.get("text", [])
            if text_output:
                print("OUTPUT:")
                print("".join(text_output)[:3000])


**Step 17A — Verify the notebook path and locate relevant cells**

In [19]:

# STEP 17A — Diagnose notebook search

import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive")

# Search for the exact notebook filename
matches = list(ROOT.rglob(
    "03_diavigil_nested_cv_and_probability_calibration.ipynb"
))

print("Matching Notebook 03 files:", len(matches))

for path in matches:
    print("Found:", path)

if not matches:
    print("\nNotebook not found. Check its location and filename.")
else:
    notebook_path = matches[0]
    nb = json.loads(notebook_path.read_text(encoding="utf-8"))

    print("\nNotebook:", notebook_path.name)
    print("Total cells:", len(nb.get("cells", [])))

    terms = [
        "calibrat", "threshold", "total_cost",
        "false negative", "false positive",
        "validation", "optimal_threshold", "C_fn", "C_fp"
    ]

    found = 0

    for i, cell in enumerate(nb.get("cells", [])):
        source = "".join(cell.get("source", []))

        if any(term in source.lower() for term in terms):
            found += 1
            print(f"\n{'=' * 60}")
            print(f"CELL {i} ({cell.get('cell_type', 'unknown')})")
            print("=" * 60)
            print(source[:7000])

    print("\nRelevant cells found:", found)


Matching Notebook 03 files: 1
Found: /content/drive/MyDrive/Colab Notebooks/03_diavigil_nested_cv_and_probability_calibration.ipynb

Notebook: 03_diavigil_nested_cv_and_probability_calibration.ipynb
Total cells: 2

Relevant cells found: 0


**Step 17B — Find the notebook containing the actual modeling code**

In [20]:

# STEP 17B — Find the actual calibration notebook

import json
from pathlib import Path

NOTEBOOK_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks"
)

terms = [
    "CalibratedClassifierCV",
    "cal_val_probs",
    "optimal_threshold",
    "total_costs",
    "threshold_grid",
    "Minimum Clinical Misclassification Cost",
    "Isotonic Probability Calibrator",
]

notebooks = list(NOTEBOOK_DIR.rglob("*.ipynb"))
print("Notebooks found:", len(notebooks))

matches = []

for path in notebooks:
    try:
        nb = json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        print("READ ERROR:", path.name, repr(e))
        continue

    cells = nb.get("cells", [])
    sources = [
        "".join(cell.get("source", []))
        for cell in cells
    ]
    all_source = "\n".join(sources)

    matched_terms = [
        term for term in terms
        if term.lower() in all_source.lower()
    ]

    if matched_terms:
        matches.append(path)
        print("\n" + "=" * 65)
        print("FILE:", path)
        print("Size (bytes):", path.stat().st_size)
        print("Cell count:", len(cells))
        print("Matched terms:", matched_terms)

        for i, source in enumerate(sources):
            if any(
                term.lower() in source.lower()
                for term in terms
            ):
                print(f"\n--- CELL {i} ---")
                print(source[:5000])

print("\nMatching notebooks:", len(matches))


Notebooks found: 16

FILE: /content/drive/MyDrive/Colab Notebooks/Fleet Maintenance & VOR Optimization Pipeline.ipynb
Size (bytes): 818780
Cell count: 100
Matched terms: ['total_costs']

--- CELL 49 ---
with engine.connect() as connection:
    print("\n--- Downtime / VOR Equivalent ---")
    vor_df = pd.read_sql_query("SELECT * FROM downtime_vor_equivalent LIMIT 10;", engine)
    display(vor_df)

    print("\n--- Dynamic Repair Costs (Sample) ---")
    costs_df = pd.read_sql_query("SELECT * FROM dynamic_repair_costs LIMIT 10;", engine)
    display(costs_df)

    print("\n--- Total Estimated Dynamic Repair Costs ---")
    total_costs_df = pd.read_sql_query("SELECT SUM(estimated_repair_cost) AS total_estimated_cost FROM dynamic_repair_costs;", engine)
    display(total_costs_df)


FILE: /content/drive/MyDrive/Colab Notebooks/Agentic_AI_Project_Diabetic_Readmission_Risk_&_Clinical_BI_Copilot (1).ipynb
Size (bytes): 1294262
Cell count: 69
Matched terms: ['CalibratedClassifierCV']

--- CELL

**Step 18 — Search all of DiaVigil's Google Drive files for the original code**

In [21]:

# STEP 18 — Recover alternate copies of DiaVigil's modeling code

import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive")

search_terms = [
    "class ClinicalDomainFeatureEngineer",
    "cal_val_probs",
    "threshold_grid",
    "optimal_threshold",
    "Minimum Clinical Misclassification Cost",
    "Fitting Isotonic Probability Calibrator",
    "C_fn = 5.0",
]

extensions = {".ipynb", ".py"}
matches = []

for path in ROOT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in extensions:
        continue

    try:
        if path.suffix.lower() == ".ipynb":
            nb = json.loads(path.read_text(encoding="utf-8"))
            sources = [
                (i, "".join(cell.get("source", [])))
                for i, cell in enumerate(nb.get("cells", []))
            ]
        else:
            sources = [(None, path.read_text(encoding="utf-8"))]

        for cell_index, source in sources:
            found = [
                term for term in search_terms
                if term.lower() in source.lower()
            ]

            if found:
                matches.append((path, cell_index, found, source))

    except Exception as e:
        # Skip unreadable files; do not stop the search.
        continue

print("Matching file/cell entries:", len(matches))

for path, cell_index, found, source in matches:
    print("\n" + "=" * 75)
    print("FILE:", path)
    print("CELL:", cell_index)
    print("MATCHES:", found)
    print("-" * 75)
    print(source[:5000])

# Also report likely backup or alternate filenames.
print("\n--- POSSIBLE NOTEBOOK BACKUPS ---")
for path in ROOT.rglob("*"):
    if path.is_file() and "diavigil" in path.name.lower():
        if any(term in path.name.lower() for term in
               ["calibration", "nested_cv", "reproducibility", "backup"]):
            print(path)


Matching file/cell entries: 19

FILE: /content/drive/MyDrive/Colab Notebooks/05_diavigil_agentic_triage_and_faithfulness_critic.ipynb
CELL: 2
MATCHES: ['class ClinicalDomainFeatureEngineer', 'optimal_threshold']
---------------------------------------------------------------------------
from google.colab import drive
import os
import joblib
import pandas as pd
import numpy as np
import re
from sklearn.base import BaseEstimator, TransformerMixin

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/DiaVigil'
DATA_PROCESSED = os.path.join(BASE_DIR, 'data/processed')

# Install explainability and typing dependencies
!pip install -q shap pydantic
import shap
from pydantic import BaseModel, Field
from typing import List, Literal

# Re-declare custom transformer classes for pipeline deserialization
ELIXHAUSER_ICD9_RULES = {
    'elix_congestive_heart_failure': r'^(39891|40201|40211|40291|40401|40403|40411|40413|40491|40493|428)',
    'elix_cardiac_arrhythmias': r'^(42610|42611|4

**Step 20: Reproduce the untouched holdout using the newly saved artifacts**

In [22]:

# STEP 20 — Re-evaluate the untouched holdout using current artifacts

import os
import shutil
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
)

ARTIFACT_DIR = "/content/drive/MyDrive/DiaVigil/data/processed"

model_path = os.path.join(
    ARTIFACT_DIR, "model_calibration_artifacts.joblib"
)
holdout_path = os.path.join(
    ARTIFACT_DIR, "audit_test_cohort.parquet"
)

# Preserve the current artifact before further changes.
backup_path = os.path.join(
    ARTIFACT_DIR, "model_calibration_artifacts_step20_backup.joblib"
)
if not os.path.exists(backup_path):
    shutil.copy2(model_path, backup_path)

# Load the artifacts currently saved by Notebook 03.
artifacts = joblib.load(model_path)
pipeline = artifacts["fitted_pipeline"]
calibrator = artifacts["calibrator"]
threshold = float(artifacts["optimal_threshold"])

# Load the untouched holdout cohort.
holdout = pd.read_parquet(holdout_path).reset_index(drop=True)

drop_cols = ["target", "encounter_id", "patient_nbr"]
X_holdout = holdout.drop(columns=drop_cols, errors="ignore")
y_test = holdout["target"].to_numpy(dtype=int)

# Transform only; never fit on the holdout.
X_transformed = pipeline.transform(X_holdout)
test_cal_probs = calibrator.predict_proba(X_transformed)[:, 1]
test_preds = (test_cal_probs >= threshold).astype(int)

# Report reproducible holdout performance.
print("Holdout rows:", len(holdout))
print("Transformed feature shape:", X_transformed.shape)
print("Saved threshold:", threshold)
print("Positive predictions:", int(test_preds.sum()))
print("Observed positive outcomes:", int(y_test.sum()))

print("\nPR-AUC:", average_precision_score(y_test, test_cal_probs))
print("ROC-AUC:", roc_auc_score(y_test, test_cal_probs))
print("Brier score:", brier_score_loss(y_test, test_cal_probs))
print("\nConfusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_test, test_preds))
print("\nClassification report:")
print(classification_report(y_test, test_preds, digits=4))

# Save a new result file without overwriting the previous audit.
new_audit = {
    "y_test": y_test,
    "test_cal_probs": test_cal_probs,
    "test_preds": test_preds,
    "optimal_threshold": threshold,
    "n_holdout": len(holdout),
    "artifact_source": "model_calibration_artifacts.joblib",
}

new_audit_path = os.path.join(
    ARTIFACT_DIR, "fairness_audit_results_reproduced_v2.joblib"
)
joblib.dump(new_audit, new_audit_path)

print("\nSaved new audit:", new_audit_path)


Holdout rows: 19875
Transformed feature shape: (19875, 98)
Saved threshold: 0.162020202020202
Positive predictions: 2742
Observed positive outcomes: 2240

PR-AUC: 0.20359523855414996
ROC-AUC: 0.6641228001336626
Brier score: 0.09615431105958255

Confusion matrix [ [TN, FP], [FN, TP] ]:
[[15546  2089]
 [ 1587   653]]

Classification report:
              precision    recall  f1-score   support

           0     0.9074    0.8815    0.8943     17635
           1     0.2381    0.2915    0.2621      2240

    accuracy                         0.8150     19875
   macro avg     0.5728    0.5865    0.5782     19875
weighted avg     0.8319    0.8150    0.8230     19875


Saved new audit: /content/drive/MyDrive/DiaVigil/data/processed/fairness_audit_results_reproduced_v2.joblib


**Step 21 — Run the final baseline fairness and calibration audit**

In [23]:

# STEP 21 — Independent subgroup fairness and calibration audit

import os
import joblib
import numpy as np
import pandas as pd

ARTIFACT_DIR = "/content/drive/MyDrive/DiaVigil/data/processed"

holdout = pd.read_parquet(
    os.path.join(ARTIFACT_DIR, "audit_test_cohort.parquet")
).reset_index(drop=True)

audit = joblib.load(
    os.path.join(
        ARTIFACT_DIR,
        "fairness_audit_results_reproduced_v2.joblib"
    )
)

y = np.asarray(audit["y_test"], dtype=int)
p = np.asarray(audit["test_cal_probs"], dtype=float)
pred = np.asarray(audit["test_preds"], dtype=int)

# Confirm alignment before subgroup analysis.
assert len(holdout) == len(y) == len(p) == len(pred)
assert np.array_equal(holdout["target"].to_numpy(), y)
assert np.array_equal(
    pred, (p >= float(audit["optimal_threshold"])).astype(int)
)

df = holdout.copy()
df["audit_y"] = y
df["audit_p"] = p
df["audit_pred"] = pred

print("Alignment verified.")
print("Holdout encounters:", len(df))
print("Threshold:", audit["optimal_threshold"])

# Wilson interval for a binomial proportion.
def wilson_interval(successes, n, z=1.96):
    if n == 0:
        return np.nan, np.nan

    rate = successes / n
    denominator = 1 + z**2 / n
    center = (rate + z**2 / (2 * n)) / denominator
    margin = (
        z * np.sqrt(
            rate * (1 - rate) / n + z**2 / (4 * n**2)
        ) / denominator
    )
    return max(0.0, center - margin), min(1.0, center + margin)


# Subgroup FNR and confidence intervals.
for col in ["age", "gender", "race"]:
    if col not in df.columns:
        print(f"\nSkipping {col}: column not present.")
        continue

    print(f"\n{'=' * 20} {col.upper()} FNR AUDIT {'=' * 20}")
    rows = []

    for group, part in df.groupby(
        col, dropna=False, observed=True
    ):
        positives = int((part["audit_y"] == 1).sum())
        fn = int(
            ((part["audit_y"] == 1) &
             (part["audit_pred"] == 0)).sum()
        )

        if positives == 0:
            fnr = low = high = np.nan
        else:
            fnr = fn / positives
            low, high = wilson_interval(fn, positives)

        rows.append({
            "group": str(group),
            "encounters": len(part),
            "positive_outcomes": positives,
            "false_negatives": fn,
            "FNR": fnr,
            "FNR_CI95_low": low,
            "FNR_CI95_high": high,
            "at_least_50_positives": positives >= 50,
        })

    result = pd.DataFrame(rows)
    display(result)

    # Primary descriptive gap: groups with at least 50 positives.
    eligible = result[
        (result["positive_outcomes"] >= 50) &
        result["FNR"].notna()
    ]

    if len(eligible) >= 2:
        high_group = eligible.loc[eligible["FNR"].idxmax()]
        low_group = eligible.loc[eligible["FNR"].idxmin()]

        print(
            "FNR gap among groups with >=50 positives:",
            round(high_group["FNR"] - low_group["FNR"], 6)
        )
        print("Highest FNR group:", high_group["group"])
        print("Lowest FNR group:", low_group["group"])
    else:
        print("Insufficient eligible groups to calculate a gap.")

# Equal-frequency calibration audit.
print("\n========== HOLDOUT CALIBRATION ==========")

calibration_df = pd.DataFrame({
    "y": y,
    "p": p,
})

calibration_df["bin"] = pd.qcut(
    calibration_df["p"],
    q=10,
    labels=False,
    duplicates="drop"
)

calibration_rows = []
ece = 0.0

for bin_id, part in calibration_df.groupby("bin", observed=True):
    mean_predicted = part["p"].mean()
    observed_rate = part["y"].mean()
    n = len(part)

    ece += (n / len(calibration_df)) * abs(
        mean_predicted - observed_rate
    )

    calibration_rows.append({
        "bin": int(bin_id) + 1,
        "n": n,
        "mean_predicted_risk": mean_predicted,
        "observed_rate": observed_rate,
        "absolute_gap": abs(mean_predicted - observed_rate),
    })

calibration_table = pd.DataFrame(calibration_rows)
display(calibration_table)
print("Equal-frequency ECE:", round(ece, 6))

# Save the tables as separate, versioned outputs.
calibration_table.to_csv(
    os.path.join(ARTIFACT_DIR, "holdout_calibration_v2.csv"),
    index=False
)

print("\nStep 21 completed.")


Alignment verified.
Holdout encounters: 19875
Threshold: 0.162020202020202

==================== AGE FNR AUDIT ====================


,group,encounters,positive_outcomes,false_negatives,FNR,FNR_CI95_low,FNR_CI95_high,at_least_50_positives
0,[0-10),29,1,1,1.000000,0.206543,1.000000,False
1,[10-20),128,11,10,0.909091,0.622635,0.983768,False
2,[20-30),307,47,31,0.659574,0.516707,0.778327,False
3,[30-40),767,89,44,0.494382,0.392913,0.596316,True
4,[40-50),1893,191,125,0.654450,0.584561,0.718249,True
5,[50-60),3431,341,224,0.656891,0.605005,0.705282,True
6,[60-70),4433,486,357,0.734568,0.693581,0.771876,True
7,[70-80),5041,603,430,0.713101,0.675739,0.747765,True
8,[80-90),3328,413,315,0.762712,0.719379,0.801203,True
9,[90-100),518,58,50,0.862069,0.750739,0.928416,True


FNR gap among groups with >=50 positives: 0.367687
Highest FNR group: [90-100)
Lowest FNR group: [30-40)

==================== GENDER FNR AUDIT ====================


,group,encounters,positive_outcomes,false_negatives,FNR,FNR_CI95_low,FNR_CI95_high,at_least_50_positives
0,Female,10792,1188,837,0.704545,0.677975,0.729797,True
1,Male,9083,1052,750,0.712928,0.684854,0.739452,True


FNR gap among groups with >=50 positives: 0.008382
Highest FNR group: Male
Lowest FNR group: Female

==================== RACE FNR AUDIT ====================


,group,encounters,positive_outcomes,false_negatives,FNR,FNR_CI95_low,FNR_CI95_high,at_least_50_positives
0,?,418,24,22,0.916667,0.741508,0.976842,False
1,AfricanAmerican,3787,417,291,0.697842,0.652126,0.739945,True
2,Asian,145,13,8,0.615385,0.355225,0.822906,False
3,Caucasian,14858,1731,1235,0.713460,0.691706,0.734270,True
4,Hispanic,376,33,18,0.545455,0.379857,0.701573,False
5,Other,291,22,13,0.590909,0.387345,0.767444,False


FNR gap among groups with >=50 positives: 0.015619
Highest FNR group: Caucasian
Lowest FNR group: AfricanAmerican

========== HOLDOUT CALIBRATION ==========


,bin,n,mean_predicted_risk,observed_rate,absolute_gap
0,1,2092,0.041148,0.042543,0.001395
1,2,2227,0.061121,0.061518,0.000397
2,3,2422,0.071803,0.065235,0.006568
3,4,1236,0.072289,0.085761,0.013472
4,5,2173,0.094724,0.089277,0.005446
5,6,2637,0.113240,0.109594,0.003645
6,7,2924,0.133393,0.133037,0.000356
7,8,741,0.153394,0.152497,0.000897
8,9,1455,0.169245,0.171134,0.001889
9,10,1968,0.265950,0.262195,0.003755


Equal-frequency ECE: 0.003504

Step 21 completed.


**STEP 22 — Patient-clustered bootstrap for observed FNR gaps**

In [24]:

# STEP 22 — Optimized patient-clustered bootstrap
# Resample patient-level counts instead of repeatedly copying encounter data.

import os
import numpy as np
import pandas as pd

N_BOOT = 1000
rng = np.random.default_rng(42)

comparisons = [
    ("age", "[90-100)", "[30-40)"),
    ("gender", "Male", "Female"),
    ("race", "Caucasian", "AfricanAmerican"),
]

assert "patient_nbr" in df.columns
assert df["patient_nbr"].notna().all()

# Create encounter-level false-negative indicator.
df = df.copy()
df["_audit_fn"] = (
    (df["audit_y"] == 1) & (df["audit_pred"] == 0)
).astype(np.int8)

# Every patient is sampled as a cluster.
patient_ids = df["patient_nbr"].drop_duplicates().to_numpy()
n_patients = len(patient_ids)

print("Unique patients:", n_patients)
print("Bootstrap repetitions:", N_BOOT)

results = []

for col, high_group, low_group in comparisons:
    if col not in df.columns:
        print(f"Skipping {col}: column missing.")
        continue

    group_stats = {}

    for group in [high_group, low_group]:
        subset = df[df[col].astype(str) == group]

        # Aggregate counts once per patient.
        agg = subset.groupby("patient_nbr").agg(
            positives=("audit_y", "sum"),
            false_negatives=("_audit_fn", "sum"),
        )

        agg = agg.reindex(patient_ids, fill_value=0)

        group_stats[group] = (
            agg["positives"].to_numpy(dtype=np.int64),
            agg["false_negatives"].to_numpy(dtype=np.int64),
        )

    pos_hi, fn_hi = group_stats[high_group]
    pos_lo, fn_lo = group_stats[low_group]

    if pos_hi.sum() == 0 or pos_lo.sum() == 0:
        print(f"Skipping {col}: comparison group has no positives.")
        continue

    observed_gap = (
        fn_hi.sum() / pos_hi.sum()
        - fn_lo.sum() / pos_lo.sum()
    )

    boot_gaps = np.empty(N_BOOT, dtype=float)

    for b in range(N_BOOT):
        # Draw patients with replacement using multiplicities.
        sampled = rng.integers(0, n_patients, size=n_patients)
        counts = np.bincount(sampled, minlength=n_patients)

        hi_pos = np.dot(counts, pos_hi)
        lo_pos = np.dot(counts, pos_lo)
        hi_fn = np.dot(counts, fn_hi)
        lo_fn = np.dot(counts, fn_lo)

        if hi_pos == 0 or lo_pos == 0:
            boot_gaps[b] = np.nan
        else:
            boot_gaps[b] = hi_fn / hi_pos - lo_fn / lo_pos

    valid = boot_gaps[np.isfinite(boot_gaps)]
    low_ci, high_ci = np.percentile(valid, [2.5, 97.5])

    results.append({
        "attribute": col,
        "higher_FNR_group": high_group,
        "lower_FNR_group": low_group,
        "observed_gap_pp": observed_gap * 100,
        "bootstrap_CI95_low_pp": low_ci * 100,
        "bootstrap_CI95_high_pp": high_ci * 100,
        "valid_replicates": len(valid),
    })

    print(
        f"{col}: gap={observed_gap * 100:.2f} percentage points; "
        f"95% CI [{low_ci * 100:.2f}, {high_ci * 100:.2f}]"
    )

bootstrap_table = pd.DataFrame(results)
display(bootstrap_table)

output_path = (
    "/content/drive/MyDrive/DiaVigil/data/processed/"
    "fairness_gap_patient_bootstrap_v1.csv"
)

bootstrap_table.to_csv(output_path, index=False)
print("\nSaved:", output_path)
print("Step 22 completed.")


Unique patients: 14022
Bootstrap repetitions: 1000
age: gap=36.77 percentage points; 95% CI [21.19, 52.01]
gender: gap=0.84 percentage points; 95% CI [-4.12, 5.58]
race: gap=1.56 percentage points; 95% CI [-4.03, 7.20]


,attribute,higher_FNR_group,lower_FNR_group,observed_gap_pp,bootstrap_CI95_low_pp,bootstrap_CI95_high_pp,valid_replicates
0,age,[90-100),[30-40),36.768694,21.191192,52.009934,1000
1,gender,Male,Female,0.838230,-4.118758,5.576244,1000
2,race,Caucasian,AfricanAmerican,1.561870,-4.033713,7.198894,1000



Saved: /content/drive/MyDrive/DiaVigil/data/processed/fairness_gap_patient_bootstrap_v1.csv
Step 22 completed.


**Step 23A — Create three separate development partitions**

In [25]:

# STEP 23A — Leakage-safe, patient-grouped development split

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# Fallback to load the cohort if it is not in the global namespace under 'dev_df' or 'dev'
if "dev_df" in globals():
    working_df = dev_df
elif "dev" in globals():
    working_df = dev
else:
    print("Loading development cohort from file...")
    working_df = pd.read_parquet(PROCESSED / "dev_cohort.parquet")

assert {"target", "patient_nbr"}.issubset(working_df.columns)

y = working_df["target"].astype(int).to_numpy()
groups = working_df["patient_nbr"].to_numpy()

# Assign every encounter to exactly one patient-grouped fold.
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_id = np.full(len(working_df), -1, dtype=int)

for fold, (_, validation_idx) in enumerate(
    splitter.split(np.zeros(len(working_df)), y, groups)
):
    fold_id[validation_idx] = fold

assert (fold_id >= 0).all(), "Some encounters were not assigned."
assert set(np.unique(fold_id)) == {0, 1, 2, 3, 4}

# Separate training, calibration, and threshold selection.
train_df = working_df.loc[fold_id < 3].copy()
calibration_df = working_df.loc[fold_id == 3].copy()
threshold_df = working_df.loc[fold_id == 4].copy()

# Verify patient separation across all three partitions.
train_patients = set(train_df["patient_nbr"])
cal_patients = set(calibration_df["patient_nbr"])
threshold_patients = set(threshold_df["patient_nbr"])

assert train_patients.isdisjoint(cal_patients)
assert train_patients.isdisjoint(threshold_patients)
assert cal_patients.isdisjoint(threshold_patients)

# Report sizes, positive rates, and patient counts.
for name, part in [
    ("Training", train_df),
    ("Calibration", calibration_df),
    ("Threshold selection", threshold_df),
]:
    print(
        f"{name}: "
        f"{len(part):,} encounters | "
        f"{part["patient_nbr"].nunique():,} patients | "
        f"positive rate = {part["target"].mean():.2%}"
    )

print("\nPatient overlap checks: ALL PASSED")
print("Step 23A completed.")

Training: 48,295 encounters | 33,857 patients | positive rate = 11.47%
Calibration: 15,849 encounters | 11,231 patients | positive rate = 11.07%
Threshold selection: 16,095 encounters | 11,329 patients | positive rate = 11.33%

Patient overlap checks: ALL PASSED
Step 23A completed.


**Step 23B — Train and calibrate the model**

In [26]:

# STEP 23B — Fit model, calibrate probabilities, select threshold

import os
import joblib
import numpy as np

from sklearn.base import clone
from sklearn.frozen import FrozenEstimator
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)
from catboost import CatBoostClassifier

ARTIFACT_DIR = "/content/drive/MyDrive/DiaVigil/data/processed"

pipeline_path = os.path.join(
    ARTIFACT_DIR, "feature_engineering_pipeline.joblib"
)

# Separate features from target and patient/encounter identifiers.
drop_cols = ["target", "encounter_id", "patient_nbr"]

def make_xy(frame):
    X = frame.drop(columns=drop_cols, errors="ignore")
    y = frame["target"].astype(int).to_numpy()
    return X, y

X_train, y_train = make_xy(train_df)
X_cal, y_cal = make_xy(calibration_df)
X_threshold, y_threshold = make_xy(threshold_df)

# Start with a fresh pipeline; fit preprocessing on training data only.
pipeline_template = joblib.load(pipeline_path)
fitted_pipeline = clone(pipeline_template)

X_train_t = fitted_pipeline.fit_transform(X_train)
X_cal_t = fitted_pipeline.transform(X_cal)
X_threshold_t = fitted_pipeline.transform(X_threshold)

print("Transformed shapes:")
print("Training:", X_train_t.shape)
print("Calibration:", X_cal_t.shape)
print("Threshold selection:", X_threshold_t.shape)

# Fit the baseline CatBoost model on training data only.
champion_model = CatBoostClassifier(
    iterations=400,
    learning_rate=0.05,
    depth=6,
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

champion_model.fit(X_train_t, y_train)

# Calibrate using a separate partition and a frozen fitted model.
calibrator = CalibratedClassifierCV(
    estimator=FrozenEstimator(champion_model),
    method="isotonic",
)
calibrator.fit(X_cal_t, y_cal)

# Choose the decision threshold on the third partition.
p_threshold = calibrator.predict_proba(X_threshold_t)[:, 1]

C_FP = 1.0
C_FN = 5.0

threshold_grid = np.linspace(0.02, 0.40, 100)
costs = []

for t in threshold_grid:
    pred = (p_threshold >= t).astype(int)
    fp = int(((pred == 1) & (y_threshold == 0)).sum())
    fn = int(((pred == 0) & (y_threshold == 1)).sum())
    costs.append(C_FP * fp + C_FN * fn)

best_idx = int(np.argmin(costs))
optimal_threshold = float(threshold_grid[best_idx])

threshold_predictions = (
    p_threshold >= optimal_threshold
).astype(int)

fp = int(
    ((threshold_predictions == 1) & (y_threshold == 0)).sum()
)
fn = int(
    ((threshold_predictions == 0) & (y_threshold == 1)).sum()
)

print("\n--- STEP 23B RESULTS ---")
print("Selected threshold:", round(optimal_threshold, 4))
print("Threshold-selection cost:", int(costs[best_idx]))
print("False positives:", fp)
print("False negatives:", fn)
print("Threshold-selection PR-AUC:",
      round(average_precision_score(y_threshold, p_threshold), 4))
print("Threshold-selection ROC-AUC:",
      round(roc_auc_score(y_threshold, p_threshold), 4))
print("Threshold-selection Brier score:",
      round(brier_score_loss(y_threshold, p_threshold), 4))

# Save separately; preserve the existing baseline artifacts.
new_artifacts = {
    "fitted_pipeline": fitted_pipeline,
    "champion_model": champion_model,
    "calibrator": calibrator,
    "optimal_threshold": optimal_threshold,
    "metadata": {
        "split_strategy": "StratifiedGroupKFold",
        "random_state": 42,
        "calibration_method": "isotonic",
        "threshold_cost_fp": C_FP,
        "threshold_cost_fn": C_FN,
        "holdout_used_for_training": False,
    },
}

output_path = os.path.join(
    ARTIFACT_DIR, "model_calibration_artifacts_step23.joblib"
)

joblib.dump(new_artifacts, output_path)

print("\nSaved new artifacts:", output_path)
print("Step 23B completed.")


Transformed shapes:
Training: (48295, 98)
Calibration: (15849, 98)
Threshold selection: (16095, 98)

--- STEP 23B RESULTS ---
Selected threshold: 0.162
Threshold-selection cost: 7922
False positives: 2107
False negatives: 1163
Threshold-selection PR-AUC: 0.2125
Threshold-selection ROC-AUC: 0.6758
Threshold-selection Brier score: 0.0958

Saved new artifacts: /content/drive/MyDrive/DiaVigil/data/processed/model_calibration_artifacts_step23.joblib
Step 23B completed.


**Step 23C — Independent holdout evaluation**

In [27]:

# STEP 23C — Independent holdout evaluation

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
    classification_report,
)

ARTIFACT_DIR = "/content/drive/MyDrive/DiaVigil/data/processed"

# Load the newly trained artifacts.
artifact_path = os.path.join(
    ARTIFACT_DIR, "model_calibration_artifacts_step23.joblib"
)
artifacts = joblib.load(artifact_path)

pipeline = artifacts["fitted_pipeline"]
model = artifacts["champion_model"]
calibrator = artifacts["calibrator"]
threshold = float(artifacts["optimal_threshold"])

# Load the untouched holdout cohort.
holdout = pd.read_parquet(
    os.path.join(ARTIFACT_DIR, "audit_test_cohort.parquet")
).reset_index(drop=True)

drop_cols = ["target", "encounter_id", "patient_nbr"]
X_test = holdout.drop(columns=drop_cols, errors="ignore")
y_test = holdout["target"].astype(int).to_numpy()

# Check patient overlap with the development partitions.
holdout_patients = set(holdout["patient_nbr"].dropna().unique())

development_patients = set(
    pd.concat([
        train_df["patient_nbr"],
        calibration_df["patient_nbr"],
        threshold_df["patient_nbr"],
    ]).dropna().unique()
)

overlap = holdout_patients.intersection(development_patients)

print("Holdout encounters:", len(holdout))
print("Unique holdout patients:", len(holdout_patients))
print("Development/holdout patient overlap:", len(overlap))

assert len(overlap) == 0, (
    "Patient overlap detected. Investigate before reporting results."
)

# Transform and predict only. Do not fit or refit.
X_test_t = pipeline.transform(X_test)
p_test = calibrator.predict_proba(X_test_t)[:, 1]
y_pred = (p_test >= threshold).astype(int)

assert len(y_test) == len(p_test) == len(y_pred)

print("\n--- STEP 23C: UNTOUCHED HOLDOUT RESULTS ---")
print("Threshold:", round(threshold, 4))
print("Holdout positive rate:", round(y_test.mean(), 4))
print("Predicted positive count:", int(y_pred.sum()))

print("PR-AUC:",
      round(average_precision_score(y_test, p_test), 4))
print("ROC-AUC:",
      round(roc_auc_score(y_test, p_test), 4))
print("Brier score:",
      round(brier_score_loss(y_test, p_test), 4))

print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))

# Save results without overwriting previous audits.
results = {
    "y_test": y_test,
    "test_cal_probs": p_test,
    "test_preds": y_pred,
    "optimal_threshold": threshold,
    "n_holdout": len(holdout),
    "artifact_source": "model_calibration_artifacts_step23.joblib",
}

results_path = os.path.join(
    ARTIFACT_DIR, "step23_holdout_evaluation.joblib"
)
joblib.dump(results, results_path)

print("\nSaved holdout evaluation:", results_path)
print("Step 23C completed.")


Holdout encounters: 19875
Unique holdout patients: 14022
Development/holdout patient overlap: 0

--- STEP 23C: UNTOUCHED HOLDOUT RESULTS ---
Threshold: 0.162
Holdout positive rate: 0.1127
Predicted positive count: 3194
PR-AUC: 0.2013
ROC-AUC: 0.6578
Brier score: 0.0961

Confusion matrix [[TN, FP], [FN, TP]]:
[[15157  2478]
 [ 1524   716]]

Classification report:
              precision    recall  f1-score   support

           0     0.9086    0.8595    0.8834     17635
           1     0.2242    0.3196    0.2635      2240

    accuracy                         0.7986     19875
   macro avg     0.5664    0.5896    0.5735     19875
weighted avg     0.8315    0.7986    0.8135     19875


Saved holdout evaluation: /content/drive/MyDrive/DiaVigil/data/processed/step23_holdout_evaluation.joblib
Step 23C completed.


**Step 23C-Check — Verify the saved predictions**

In [28]:

# STEP 23C-CHECK — Verify holdout metrics and count consistency

import os
import joblib
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

ARTIFACT_DIR = "/content/drive/MyDrive/DiaVigil/data/processed"

results_path = os.path.join(
    ARTIFACT_DIR, "step23_holdout_evaluation.joblib"
)
r = joblib.load(results_path)

y = np.asarray(r["y_test"], dtype=int)
p = np.asarray(r["test_cal_probs"], dtype=float)
pred = np.asarray(r["test_preds"], dtype=int)
tau = float(r["optimal_threshold"])

print("Array lengths:", len(y), len(p), len(pred))
print("Actual positive outcomes:", int(y.sum()))
print("Actual positive rate:", round(float(y.mean()), 6))
print("Predicted positive count:", int(pred.sum()))
print("Saved threshold:", tau)

print(
    "Predictions match saved threshold:",
    np.array_equal(pred, (p >= tau).astype(int))
)

cm = confusion_matrix(y, pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
print(cm)
print("\nTN:", tn, "FP:", fp, "FN:", fn, "TP:", tp)
print("Matrix total:", int(cm.sum()))
print("Actual class counts:", np.bincount(y, minlength=2))
print("Predicted class counts:", np.bincount(pred, minlength=2))

print("\nRecomputed classification report:")
print(classification_report(y, pred, labels=[0, 1], digits=4))

assert len(y) == len(p) == len(pred)
assert cm.sum() == len(y)
assert np.array_equal(pred, (p >= tau).astype(int))

print("\nCount and threshold consistency checks passed.")


Array lengths: 19875 19875 19875
Actual positive outcomes: 2240
Actual positive rate: 0.112704
Predicted positive count: 3194
Saved threshold: 0.162020202020202
Predictions match saved threshold: True

Confusion matrix [[TN, FP], [FN, TP]]:
[[15157  2478]
 [ 1524   716]]

TN: 15157 FP: 2478 FN: 1524 TP: 716
Matrix total: 19875
Actual class counts: [17635  2240]
Predicted class counts: [16681  3194]

Recomputed classification report:
              precision    recall  f1-score   support

           0     0.9086    0.8595    0.8834     17635
           1     0.2242    0.3196    0.2635      2240

    accuracy                         0.7986     19875
   macro avg     0.5664    0.5896    0.5735     19875
weighted avg     0.8315    0.7986    0.8135     19875


Count and threshold consistency checks passed.


**Step 24 — Holdout calibration and subgroup fairness audit**

In [29]:

# STEP 24 — Holdout calibration and subgroup false-negative audit

import os
import joblib
import numpy as np
import pandas as pd

ARTIFACT_DIR = "/content/drive/MyDrive/DiaVigil/data/processed"

results = joblib.load(
    os.path.join(ARTIFACT_DIR, "step23_holdout_evaluation.joblib")
)

holdout = pd.read_parquet(
    os.path.join(ARTIFACT_DIR, "audit_test_cohort.parquet")
).reset_index(drop=True)

y = np.asarray(results["y_test"], dtype=int)
p = np.asarray(results["test_cal_probs"], dtype=float)
pred = np.asarray(results["test_preds"], dtype=int)

# Confirm row alignment before using subgroup metadata.
assert len(holdout) == len(y) == len(p) == len(pred)
assert np.array_equal(holdout["target"].to_numpy(), y)

holdout["actual"] = y
holdout["probability"] = p
holdout["prediction"] = pred
holdout["false_negative"] = (
    (holdout["actual"] == 1) & (holdout["prediction"] == 0)
).astype(int)

# 1. Equal-frequency calibration bins.
cal = pd.DataFrame({
    "probability": p,
    "actual": y
})

cal["bin"] = pd.qcut(
    cal["probability"],
    q=10,
    labels=False,
    duplicates="drop"
)

calibration_table = (
    cal.groupby("bin", observed=True)
    .agg(
        encounters=("actual", "size"),
        mean_predicted_risk=("probability", "mean"),
        observed_rate=("actual", "mean")
    )
    .reset_index()
)

calibration_table["absolute_gap"] = (
    calibration_table["mean_predicted_risk"]
    - calibration_table["observed_rate"]
).abs()

ece = np.average(
    calibration_table["absolute_gap"],
    weights=calibration_table["encounters"]
)

print("=== HOLDOUT CALIBRATION ===")
print(calibration_table.round(5).to_string(index=False))
print("Equal-frequency ECE:", round(float(ece), 5))

# 2. Wilson 95% confidence interval for a proportion.
def wilson_ci(successes, total, z=1.959964):
    if total == 0:
        return (np.nan, np.nan)

    rate = successes / total
    denom = 1 + z**2 / total
    center = (rate + z**2 / (2 * total)) / denom
    margin = (
        z * np.sqrt(
            rate * (1 - rate) / total
            + z**2 / (4 * total**2)
        ) / denom
    )
    return max(0, center - margin), min(1, center + margin)

# 3. Subgroup false-negative rates.
# Age bins are kept as recorded in the holdout dataset.
subgroup_tables = {}

for column in ["age", "gender", "race"]:
    if column not in holdout.columns:
        print(f"\nSkipping {column}: column not present.")
        continue

    rows = []

    for group, frame in holdout.groupby(
        column, dropna=False, observed=True
    ):
        actual_positive = int((frame["actual"] == 1).sum())
        false_negatives = int(frame["false_negative"].sum())

        if actual_positive == 0:
            fnr = np.nan
            low, high = np.nan, np.nan
        else:
            fnr = false_negatives / actual_positive
            low, high = wilson_ci(
                false_negatives, actual_positive
            )

        rows.append({
            "group": str(group),
            "encounters": len(frame),
            "positive_outcomes": actual_positive,
            "false_negatives": false_negatives,
            "FNR": fnr,
            "FNR_CI95_low": low,
            "FNR_CI95_high": high,
            "at_least_50_positives": actual_positive >= 50
        })

    table = pd.DataFrame(rows).sort_values("group")
    subgroup_tables[column] = table

    print(f"\n=== {column.upper()} FALSE-NEGATIVE AUDIT ===")
    print(table.round(4).to_string(index=False))

    # Descriptive disparity among groups with enough positive outcomes.
    eligible = table[
        table["at_least_50_positives"] & table["FNR"].notna()
    ]

    if len(eligible) >= 2:
        highest = eligible.loc[eligible["FNR"].idxmax()]
        lowest = eligible.loc[eligible["FNR"].idxmin()]

        print(
            "FNR gap among groups with >=50 positive outcomes:",
            round(float(highest["FNR"] - lowest["FNR"]), 4)
        )
        print("Highest FNR group:", highest["group"])
        print("Lowest FNR group:", lowest["group"])

# 4. Save the new audit separately.
audit_output = {
    "calibration_table": calibration_table,
    "ece": float(ece),
    "subgroup_tables": subgroup_tables,
    "threshold": float(results["optimal_threshold"]),
    "n_holdout": len(holdout)
}

output_path = os.path.join(
    ARTIFACT_DIR, "step24_holdout_fairness_calibration.joblib"
)

joblib.dump(audit_output, output_path)

print("\nSaved Step 24 audit:", output_path)
print("Step 24 completed.")


=== HOLDOUT CALIBRATION ===
 bin  encounters  mean_predicted_risk  observed_rate  absolute_gap
   0        3180              0.04889        0.05346       0.00457
   1        2284              0.05665        0.06305       0.00640
   2        2293              0.07077        0.07196       0.00119
   3        1105              0.07740        0.08416       0.00677
   4        1155              0.09177        0.10043       0.00867
   5        2278              0.10423        0.10799       0.00376
   6        3543              0.13709        0.12588       0.01121
   7         143              0.14992        0.17483       0.02490
   8        2597              0.17391        0.18252       0.00861
   9        1297              0.28359        0.27833       0.00525
Equal-frequency ECE: 0.00656

=== AGE FALSE-NEGATIVE AUDIT ===
   group  encounters  positive_outcomes  false_negatives    FNR  FNR_CI95_low  FNR_CI95_high  at_least_50_positives
  [0-10)          29                  1                1

**Step 25 — Compare the baseline and new model**

In [30]:

# STEP 25 — Paired baseline vs. new-model holdout comparison

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
)

ARTIFACT_DIR = "/content/drive/MyDrive/DiaVigil/data/processed"

baseline_path = os.path.join(
    ARTIFACT_DIR, "fairness_audit_results.joblib"
)
new_path = os.path.join(
    ARTIFACT_DIR, "step23_holdout_evaluation.joblib"
)

baseline = joblib.load(baseline_path)
new = joblib.load(new_path)

holdout = pd.read_parquet(
    os.path.join(ARTIFACT_DIR, "audit_test_cohort.parquet")
).reset_index(drop=True)

y = holdout["target"].to_numpy(dtype=int)

y_base = np.asarray(baseline["y_test"], dtype=int)
p_base = np.asarray(baseline["test_cal_probs"], dtype=float)
pred_base = np.asarray(baseline["test_preds"], dtype=int)

y_new = np.asarray(new["y_test"], dtype=int)
p_new = np.asarray(new["test_cal_probs"], dtype=float)
pred_new = np.asarray(new["test_preds"], dtype=int)

# Verify alignment before comparing models.
assert len(y) == len(y_base) == len(y_new)
assert np.array_equal(y, y_base)
assert np.array_equal(y, y_new)

C_FP = 1.0
C_FN = 5.0

def evaluate(name, probs, preds):
    tn, fp, fn, tp = confusion_matrix(
        y, preds, labels=[0, 1]
    ).ravel()

    return {
        "model": name,
        "PR_AUC": average_precision_score(y, probs),
        "ROC_AUC": roc_auc_score(y, probs),
        "Brier": brier_score_loss(y, probs),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": tp / (tp + fp) if tp + fp else np.nan,
        "Recall": tp / (tp + fn) if tp + fn else np.nan,
        "FNR": fn / (fn + tp) if fn + tp else np.nan,
        "Cost_FP1_FN5": C_FP * fp + C_FN * fn,
    }

comparison = pd.DataFrame([
    evaluate("Original baseline", p_base, pred_base),
    evaluate("Step 23 model", p_new, pred_new),
])

print("=== SAME-HOLDOUT MODEL COMPARISON ===")
print(comparison.round(5).to_string(index=False))

# Compare subgroup FNRs using the same age, gender, and race labels.
for column in ["age", "gender", "race"]:
    if column not in holdout.columns:
        continue

    subgroup_results = []

    for group, indices in holdout.groupby(
        column, dropna=False, observed=True
    ).groups.items():
        idx = np.asarray(list(indices), dtype=int)
        positive = y[idx] == 1

        if positive.sum() == 0:
            continue

        fnr_base = np.mean(pred_base[idx][positive] == 0)
        fnr_new = np.mean(pred_new[idx][positive] == 0)

        subgroup_results.append({
            "group": str(group),
            "positive_outcomes": int(positive.sum()),
            "baseline_FNR": fnr_base,
            "new_FNR": fnr_new,
            "change_percentage_points": 100 * (
                fnr_new - fnr_base
            ),
        })

    print(f"\n=== {column.upper()} FNR COMPARISON ===")
    print(
        pd.DataFrame(subgroup_results)
        .round(4)
        .to_string(index=False)
    )

comparison_path = os.path.join(
    ARTIFACT_DIR, "step25_baseline_comparison.csv"
)
comparison.to_csv(comparison_path, index=False)

print("\nSaved comparison:", comparison_path)
print("Step 25 completed.")


=== SAME-HOLDOUT MODEL COMPARISON ===
            model  PR_AUC  ROC_AUC   Brier    TN   FP   FN  TP  Precision  Recall     FNR  Cost_FP1_FN5
Original baseline 0.20360  0.66412 0.09615 15546 2089 1587 653    0.23815 0.29152 0.70848       10024.0
    Step 23 model 0.20132  0.65785 0.09609 15157 2478 1524 716    0.22417 0.31964 0.68036       10098.0

=== AGE FNR COMPARISON ===
   group  positive_outcomes  baseline_FNR  new_FNR  change_percentage_points
  [0-10)                  1        1.0000   1.0000                    0.0000
 [10-20)                 11        0.9091   0.9091                    0.0000
 [20-30)                 47        0.6596   0.5319                  -12.7660
 [30-40)                 89        0.4944   0.5730                    7.8652
 [40-50)                191        0.6545   0.6963                    4.1885
 [50-60)                341        0.6569   0.6452                   -1.1730
 [60-70)                486        0.7346   0.7058                   -2.8807
 [70-8

**Step 26 — Paired patient-cluster bootstrap**

In [31]:

# STEP 26 — Paired patient-cluster bootstrap comparison

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
)

ARTIFACT_DIR = "/content/drive/MyDrive/DiaVigil/data/processed"

baseline = joblib.load(
    os.path.join(ARTIFACT_DIR, "fairness_audit_results.joblib")
)
new = joblib.load(
    os.path.join(ARTIFACT_DIR, "step23_holdout_evaluation.joblib")
)
holdout = pd.read_parquet(
    os.path.join(ARTIFACT_DIR, "audit_test_cohort.parquet")
).reset_index(drop=True)

y = holdout["target"].to_numpy(dtype=int)
patient_codes, unique_patients = pd.factorize(
    holdout["patient_nbr"], sort=False
)

p_base = np.asarray(baseline["test_cal_probs"], dtype=float)
pred_base = np.asarray(baseline["test_preds"], dtype=int)
p_new = np.asarray(new["test_cal_probs"], dtype=float)
pred_new = np.asarray(new["test_preds"], dtype=int)

assert len(y) == len(p_base) == len(p_new)
assert len(y) == len(pred_base) == len(pred_new)
assert np.array_equal(y, np.asarray(baseline["y_test"], dtype=int))
assert np.array_equal(y, np.asarray(new["y_test"], dtype=int))

# Build encounter-index clusters by patient.
cluster_indices = [
    np.flatnonzero(patient_codes == code)
    for code in range(len(unique_patients))
]

def metrics(yb, pb, pred):
    tn, fp, fn, tp = confusion_matrix(
        yb, pred, labels=[0, 1]
    ).ravel()

    n = len(yb)

    return {
        "PR_AUC": average_precision_score(yb, pb),
        "ROC_AUC": roc_auc_score(yb, pb),
        "Brier": brier_score_loss(yb, pb),
        "Recall": tp / (tp + fn) if tp + fn else np.nan,
        "FNR": fn / (tp + fn) if tp + fn else np.nan,
        "Cost_per_1000": 1000 * (fp + 5 * fn) / n,
    }

# Observed paired differences: new model minus baseline.
observed_base = metrics(y, p_base, pred_base)
observed_new = metrics(y, p_new, pred_new)

metric_names = list(observed_base)
observed_delta = {
    name: observed_new[name] - observed_base[name]
    for name in metric_names
}

N_BOOT = 1000
rng = np.random.default_rng(42)
boot_deltas = {name: [] for name in metric_names}

for b in range(N_BOOT):
    # Sample patients with replacement, retaining all their encounters.
    sampled_clusters = rng.integers(
        0, len(cluster_indices), size=len(cluster_indices)
    )
    idx = np.concatenate([
        cluster_indices[j] for j in sampled_clusters
    ])

    yb = y[idx]

    # Skip degenerate replicates without both outcome classes.
    if np.unique(yb).size < 2:
        continue

    mb = metrics(yb, p_base[idx], pred_base[idx])
    mn = metrics(yb, p_new[idx], pred_new[idx])

    for name in metric_names:
        boot_deltas[name].append(mn[name] - mb[name])

rows = []

for name in metric_names:
    values = np.asarray(boot_deltas[name], dtype=float)
    values = values[np.isfinite(values)]

    low, high = np.percentile(values, [2.5, 97.5])

    rows.append({
        "metric": name,
        "baseline": observed_base[name],
        "step23": observed_new[name],
        "difference_new_minus_baseline": observed_delta[name],
        "bootstrap_95CI_low": low,
        "bootstrap_95CI_high": high,
        "valid_replicates": len(values),
    })

summary = pd.DataFrame(rows)

print("=== PAIRED PATIENT-CLUSTER BOOTSTRAP ===")
print("Unique patient clusters:", len(cluster_indices))
print("Requested replicates:", N_BOOT)
print(summary.round(5).to_string(index=False))

output_path = os.path.join(
    ARTIFACT_DIR, "step26_paired_cluster_bootstrap.csv"
)
summary.to_csv(output_path, index=False)

print("\nSaved:", output_path)
print("Step 26 completed.")


=== PAIRED PATIENT-CLUSTER BOOTSTRAP ===
Unique patient clusters: 14022
Requested replicates: 1000
       metric  baseline    step23  difference_new_minus_baseline  bootstrap_95CI_low  bootstrap_95CI_high  valid_replicates
       PR_AUC   0.20360   0.20132                       -0.00227            -0.00807              0.00362              1000
      ROC_AUC   0.66412   0.65785                       -0.00627            -0.01091             -0.00136              1000
        Brier   0.09615   0.09609                       -0.00006            -0.00041              0.00028              1000
       Recall   0.29152   0.31964                        0.02812             0.01543              0.04162              1000
          FNR   0.70848   0.68036                       -0.02812            -0.04162             -0.01543              1000
Cost_per_1000 504.35220 508.07547                        3.72327            -4.49549             11.75015              1000

Saved: /content/drive/MyDrive/Di

**Step 27A — Generate the research audit summary**

In [32]:

# STEP 27A — Consolidated Model Audit Report

from pathlib import Path
import pandas as pd
import numpy as np

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/DiaVigil/data/processed"
)

comparison_path = ARTIFACT_DIR / "step25_baseline_comparison.csv"
bootstrap_path = ARTIFACT_DIR / "step26_paired_cluster_bootstrap.csv"

# Confirm required evidence exists
for path in [comparison_path, bootstrap_path]:
    assert path.exists(), f"Missing required artifact: {path}"

comparison = pd.read_csv(comparison_path)
bootstrap = pd.read_csv(bootstrap_path)

print("STEP 27 — DIA VIGIL RESEARCH AUDIT")
print("=" * 55)

print("\n1. BASELINE VS. STEP 23 MODEL")
display(comparison)

print("\n2. PAIRED PATIENT-CLUSTER BOOTSTRAP")
display(bootstrap)

# Interpret confidence intervals when the expected fields exist.
required = {
    "metric",
    "difference_new_minus_baseline",
    "bootstrap_95CI_low",
    "bootstrap_95CI_high",
}

if required.issubset(bootstrap.columns):
    print("\n3. STATISTICAL INTERPRETATION")

    for _, row in bootstrap.iterrows():
        metric = row["metric"]
        low = row["bootstrap_95CI_low"]
        high = row["bootstrap_95CI_high"]
        diff = row["difference_new_minus_baseline"]

        if low > 0 or high < 0:
            conclusion = "CI excludes zero"
        else:
            conclusion = "CI includes zero"

        print(
            f"{metric}: difference={diff:.6f}; "
            f"95% CI=[{low:.6f}, {high:.6f}]; "
            f"{conclusion}"
        )

# Save a consolidated evidence package without overwriting
# the original model or audit artifacts.
comparison.to_csv(
    ARTIFACT_DIR / "step27_model_comparison.csv",
    index=False
)
bootstrap.to_csv(
    ARTIFACT_DIR / "step27_bootstrap_summary.csv",
    index=False
)

print("\nStep 27A completed.")
print("Consolidated comparison and bootstrap tables saved.")



STEP 27 — DIA VIGIL RESEARCH AUDIT

1. BASELINE VS. STEP 23 MODEL


,model,PR_AUC,ROC_AUC,Brier,TN,FP,FN,TP,Precision,Recall,FNR,Cost_FP1_FN5
0,Original baseline,0.203595,0.664123,0.096154,15546,2089,1587,653,0.238147,0.291518,0.708482,10024.0
1,Step 23 model,0.201322,0.657848,0.096093,15157,2478,1524,716,0.224170,0.319643,0.680357,10098.0



2. PAIRED PATIENT-CLUSTER BOOTSTRAP


,metric,baseline,step23,difference_new_minus_baseline,bootstrap_95CI_low,bootstrap_95CI_high,valid_replicates
0,PR_AUC,0.203595,0.201322,-0.002273,-0.008068,0.003624,1000
1,ROC_AUC,0.664123,0.657848,-0.006274,-0.010908,-0.001361,1000
2,Brier,0.096154,0.096093,-0.000061,-0.000411,0.000276,1000
3,Recall,0.291518,0.319643,0.028125,0.015426,0.041615,1000
4,FNR,0.708482,0.680357,-0.028125,-0.041615,-0.015426,1000
5,Cost_per_1000,504.352201,508.075472,3.723270,-4.495490,11.750153,1000



3. STATISTICAL INTERPRETATION
PR_AUC: difference=-0.002273; 95% CI=[-0.008068, 0.003624]; CI includes zero
ROC_AUC: difference=-0.006274; 95% CI=[-0.010908, -0.001361]; CI excludes zero
Brier: difference=-0.000061; 95% CI=[-0.000411, 0.000276]; CI includes zero
Recall: difference=0.028125; 95% CI=[0.015426, 0.041615]; CI excludes zero
FNR: difference=-0.028125; 95% CI=[-0.041615, -0.015426]; CI excludes zero
Cost_per_1000: difference=3.723270; 95% CI=[-4.495490, 11.750153]; CI includes zero

Step 27A completed.
Consolidated comparison and bootstrap tables saved.


**Step 27B — Save the research decision report**

In [33]:

# STEP 27B — DiaVigil Research Decision Report

from pathlib import Path
from datetime import date

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/DiaVigil/data/processed"
)
assert ARTIFACT_DIR.exists(), "DiaVigil artifact directory not found."

report = f"""# DiaVigil: Model Audit and Selection Report

**Report date:** {date.today().isoformat()}
**Evaluation cohort:** 19,875 encounters
**Bootstrap procedure:** Paired patient-cluster bootstrap
**Valid bootstrap replicates:** 1,000

## 1. Research objective

Evaluate the trade-off between predictive discrimination, detection
of positive readmission outcomes, probability calibration, and
cost-sensitive classification for the original baseline and Step 23 model.

## 2. Holdout results

| Metric | Original baseline | Step 23 model |
|---|---:|---:|
| PR-AUC | 0.203595 | 0.201322 |
| ROC-AUC | 0.664123 | 0.657848 |
| Brier score | 0.096154 | 0.096093 |
| Precision | 0.238147 | 0.224170 |
| Recall | 0.291518 | 0.319643 |
| False-negative rate | 0.708482 | 0.680357 |
| False positives | 2,089 | 2,478 |
| False negatives | 1,587 | 1,524 |
| True positives | 653 | 716 |
| True negatives | 15,546 | 15,157 |
| Assumed total error cost | 10,024 | 10,098 |

## 3. Paired patient-cluster bootstrap findings

Differences below are Step 23 minus baseline.

| Metric | Difference | 95% bootstrap interval |
|---|---:|---:|
| PR-AUC | -0.002273 | [-0.008068, 0.003624] |
| ROC-AUC | -0.006274 | [-0.010908, -0.001361] |
| Brier score | -0.000061 | [-0.000411, 0.000276] |
| Recall | +0.028125 | [+0.015426, +0.041615] |
| False-negative rate | -0.028125 | [-0.041615, -0.015426] |
| Cost per 1,000 encounters | +3.723270 | [-4.495490, +11.750153] |

The recall improvement and reduction in false-negative rate have
bootstrap intervals excluding zero. The ROC-AUC difference also
excludes zero and favors the baseline. PR-AUC, Brier score, and
cost differences have intervals including zero.

## 4. Interpretation

Step 23 identifies more positive outcomes at its selected threshold,
but also produces more false-positive classifications. Its improved
recall is accompanied by lower ROC-AUC and precision.

The observed cost is slightly higher under the specified cost ratio
of 1 per false positive and 5 per false negative. The bootstrap
interval does not establish a reliable cost advantage for either model.

Neither model should be declared the universal winner from these
results alone. Model selection depends on the intended clinical
workflow, capacity for follow-up, and consequences of missed cases.

## 5. Limitations and next validation requirements

- The holdout has already informed multiple analyses and should not
  be described as a newly untouched final test set.
- Bootstrap intervals quantify sampling uncertainty under the
  resampling procedure; they do not establish clinical effectiveness.
- Subgroup fairness findings require attention to sample size,
  uncertainty, missing demographic labels, and multiple comparisons.
- Calibration should be assessed overall and within relevant groups.
- The cost ratio is an assumption, not an established estimate of
  actual hospital costs or savings.
- Independent temporal or external validation is needed before
  making claims about generalization to other hospitals or populations.
- DiaVigil's agentic recommendations require separate evaluation of
  clinical faithfulness, safety, and human oversight.

## 6. Provisional research decision

Retain both models as comparison candidates. Document the recall
versus discrimination trade-off and complete the reproducibility,
subgroup fairness, calibration, and clinical-utility review before
making a final selection.

DiaVigil is a research prototype and is not validated for autonomous
clinical decision-making.

---
Generated from the Step 27A audit results.
"""

report_path = ARTIFACT_DIR / "step27_research_audit_report.md"
report_path.write_text(report, encoding="utf-8")

print("Step 27B completed.")
print("Research report saved to:", report_path)
print("Report size:", report_path.stat().st_size, "bytes")


Step 27B completed.
Research report saved to: /content/drive/MyDrive/DiaVigil/data/processed/step27_research_audit_report.md
Report size: 3555 bytes


**Step 28A — Audit your existing research artifacts**

In [34]:

# STEP 28A — DiaVigil Research Artifact Inventory

from pathlib import Path
import pandas as pd

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/DiaVigil/data/processed"
)

expected_files = [
    "dev_cohort.parquet",
    "audit_test_cohort.parquet",
    "feature_engineering_pipeline.joblib",
    "model_calibration_artifacts.joblib",
    "fairness_audit_results.joblib",
    "fairness_gap_patient_bootstrap_v1.csv",
    "simulation_artifacts.joblib",
    "agent_audit_demonstration.joblib",
    "step23_holdout_evaluation.joblib",
    "step24_holdout_fairness_calibration.joblib",
    "step25_baseline_comparison.csv",
    "step26_paired_cluster_bootstrap.csv",
    "step27_model_comparison.csv",
    "step27_bootstrap_summary.csv",
    "step27_research_audit_report.md",
]

records = []

for filename in expected_files:
    path = ARTIFACT_DIR / filename

    records.append({
        "Artifact": filename,
        "Exists": path.exists(),
        "Size (bytes)": path.stat().st_size if path.exists() else None
    })

inventory = pd.DataFrame(records)

print("STEP 28A — RESEARCH ARTIFACT INVENTORY")
print("=" * 55)
display(inventory)

print("\nPresent:", int(inventory["Exists"].sum()),
      "of", len(inventory))
print("Missing:", inventory.loc[~inventory["Exists"], "Artifact"].tolist())
print("\nStep 28A completed. No artifacts were modified.")


STEP 28A — RESEARCH ARTIFACT INVENTORY


,Artifact,Exists,Size (bytes)
0,dev_cohort.parquet,True,2192155
1,audit_test_cohort.parquet,True,566805
2,feature_engineering_pipeline.joblib,True,2346
3,model_calibration_artifacts.joblib,True,468095
4,fairness_audit_results.joblib,True,481610
5,fairness_gap_patient_bootstrap_v1.csv,True,376
6,simulation_artifacts.joblib,True,3007
7,agent_audit_demonstration.joblib,True,1318
8,step23_holdout_evaluation.joblib,True,477533
9,step24_holdout_fairness_calibration.joblib,True,5437



Present: 15 of 15
Missing: []

Step 28A completed. No artifacts were modified.


**Step 28B — Verify artifact integrity and research evidence**

In [35]:

# STEP 28B — Inspect Saved DiaVigil Research Artifacts

from pathlib import Path
import joblib
import pandas as pd
import numpy as np

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/DiaVigil/data/processed"
)

files_to_inspect = [
    "model_calibration_artifacts.joblib",
    "fairness_audit_results.joblib",
    "simulation_artifacts.joblib",
    "agent_audit_demonstration.joblib",
    "step23_holdout_evaluation.joblib",
    "step24_holdout_fairness_calibration.joblib",
]

def describe_object(obj, indent=0):
    prefix = " " * indent

    if isinstance(obj, dict):
        print(f"{prefix}Dictionary keys: {list(obj.keys())}")
        for key, value in obj.items():
            print(f"{prefix}- {key}: ", end="")
            if isinstance(value, pd.DataFrame):
                print(
                    f"DataFrame, shape={value.shape}, "
                    f"columns={list(value.columns)}"
                )
            elif isinstance(value, np.ndarray):
                print(f"Array, shape={value.shape}, dtype={value.dtype}")
            elif isinstance(value, dict):
                print("Nested dictionary")
                describe_object(value, indent + 2)
            elif isinstance(value, (list, tuple)):
                print(f"{type(value).__name__}, length={len(value)}")
            else:
                print(type(value).__name__)
    elif isinstance(obj, pd.DataFrame):
        print("DataFrame:", obj.shape, list(obj.columns))
    else:
        print("Object type:", type(obj).__name__)

print("STEP 28B — ARTIFACT CONTENT INSPECTION")
print("=" * 60)

inspection_results = []

for filename in files_to_inspect:
    path = ARTIFACT_DIR / filename
    print(f"\n--- {filename} ---")

    if not path.exists():
        print("MISSING")
        inspection_results.append((filename, "MISSING"))
        continue

    try:
        artifact = joblib.load(path)
        describe_object(artifact)
        inspection_results.append((filename, "LOADED"))
    except Exception as exc:
        print("LOAD ERROR:", type(exc).__name__, str(exc))
        inspection_results.append(
            (filename, f"ERROR: {type(exc).__name__}")
        )

print("\n--- INSPECTION SUMMARY ---")
display(pd.DataFrame(
    inspection_results,
    columns=["Artifact", "Inspection Status"]
))

print("\nStep 28B completed.")
print("Inspection does not modify the saved artifacts.")


STEP 28B — ARTIFACT CONTENT INSPECTION

--- model_calibration_artifacts.joblib ---
Dictionary keys: ['fitted_pipeline', 'champion_model', 'calibrator', 'optimal_threshold']
- fitted_pipeline: Pipeline
- champion_model: CatBoostClassifier
- calibrator: CalibratedClassifierCV
- optimal_threshold: float64

--- fairness_audit_results.joblib ---
Dictionary keys: ['age_audit', 'gender_audit', 'race_audit', 'test_cal_probs', 'test_preds', 'y_test']
- age_audit: DataFrame, shape=(10, 9), columns=['Subgroup', 'Encounters', 'Readmitted (<30d)', 'Base Prevalence', 'Mean Pred Risk', 'Selection Rate %', 'Sensitivity (TPR)', 'False Neg Rate (FNR)', 'False Pos Rate (FPR)']
- gender_audit: DataFrame, shape=(2, 9), columns=['Subgroup', 'Encounters', 'Readmitted (<30d)', 'Base Prevalence', 'Mean Pred Risk', 'Selection Rate %', 'Sensitivity (TPR)', 'False Neg Rate (FNR)', 'False Pos Rate (FPR)']
- race_audit: DataFrame, shape=(6, 9), columns=['Subgroup', 'Encounters', 'Readmitted (<30d)', 'Base Prevalenc

,Artifact,Inspection Status
0,model_calibration_artifacts.joblib,LOADED
1,fairness_audit_results.joblib,LOADED
2,simulation_artifacts.joblib,LOADED
3,agent_audit_demonstration.joblib,LOADED
4,step23_holdout_evaluation.joblib,LOADED
5,step24_holdout_fairness_calibration.joblib,LOADED



Step 28B completed.
Inspection does not modify the saved artifacts.


**Step 28C — Cross-check the saved evidence**

In [36]:
# STEP 28C — Cross-Artifact Consistency Checks

from pathlib import Path
import joblib
import pandas as pd
import numpy as np

D = Path("/content/drive/MyDrive/DiaVigil/data/processed")

# Load existing artifacts
baseline = joblib.load(D / "fairness_audit_results.joblib")
step23 = joblib.load(D / "step23_holdout_evaluation.joblib")
fairness = joblib.load(D / "step24_holdout_fairness_calibration.joblib")
agent = joblib.load(D / "agent_audit_demonstration.joblib")
holdout = pd.read_parquet(D / "audit_test_cohort.parquet").reset_index(drop=True)

checks = []

def record(name, passed, details):
    checks.append({
        "Check": name,
        "Passed": bool(passed),
        "Details": str(details)
    })

# 1. Holdout size and label alignment
y = holdout["target"].to_numpy(dtype=int)
y_base = np.asarray(baseline["y_test"], dtype=int)
y_new = np.asarray(step23["y_test"], dtype=int)

record(
    "Holdout and label lengths",
    len(y) == len(y_base) == len(y_new),
    f"holdout={len(y)}, baseline={len(y_base)}, Step23={len(y_new)}"
)

record(
    "Baseline labels match holdout",
    np.array_equal(y, y_base),
    "Compared in current row order"
)

record(
    "Step23 labels match holdout",
    np.array_equal(y, y_new),
    "Compared in current row order"
)

# 2. Saved predictions agree with their probabilities and thresholds
for name, artifact in [("Baseline", baseline), ("Step23", step23)]:
    probs = np.asarray(artifact["test_cal_probs"], dtype=float)
    preds = np.asarray(artifact["test_preds"], dtype=int)

    # Fallback if 'optimal_threshold' key is not present in the artifact dictionary
    if "optimal_threshold" in artifact:
        tau = float(artifact["optimal_threshold"])
    else:
        # Original baseline decision threshold was established at 0.162
        tau = 0.162020202020202

    lengths_ok = len(probs) == len(preds) == len(y)
    record(
        f"{name} prediction lengths",
        lengths_ok,
        f"probabilities={len(probs)}, predictions={len(preds)}"
    )

    if lengths_ok:
        record(
            f"{name} threshold consistency",
            np.array_equal(preds, (probs >= tau).astype(int)),
            f"Saved threshold={tau:.6f}"
        )

    record(
        f"{name} probabilities valid",
        np.isfinite(probs).all() and
        ((probs >= 0) & (probs <= 1)).all(),
        "All probabilities should be finite and in [0, 1]"
    )

# 3. Fairness/calibration artifact dimensions and threshold
record(
    "Fairness holdout count",
    int(fairness["n_holdout"]) == len(y),
    f"Saved n_holdout={fairness['n_holdout']}; actual={len(y)}"
)

record(
    "Fairness threshold matches Step23",
    np.isclose(
        float(fairness["threshold"]),
        float(step23["optimal_threshold"])
    ),
    f"Fairness={float(fairness['threshold']):.6f}; "
    f"Step23={float(step23['optimal_threshold']):.6f}"
)

for group_name, table in fairness["subgroup_tables"].items():
    record(
        f"{group_name} subgroup table available",
        isinstance(table, pd.DataFrame) and not table.empty,
        f"Rows={len(table)}"
    )

# 4. Agent audit structure and decision flags
for key in ["sample_passed_brief", "sample_rejected_brief"]:
    brief = agent.get(key)
    record(
        f"Agent example {key} exists",
        isinstance(brief, dict),
        f"Available={isinstance(brief, dict)}"
    )

if isinstance(agent.get("sample_passed_brief"), dict):
    record(
        "Accepted example critic flag",
        agent["sample_passed_brief"].get("critic_audit_passed") is True,
        "Checks saved audit decision"
    )

if isinstance(agent.get("sample_rejected_brief"), dict):
    record(
        "Rejected example critic flag",
        agent["sample_rejected_brief"].get("critic_audit_passed") is False,
        "Checks saved audit decision"
    )

# Report results; no patient identifiers or individual records printed.
results = pd.DataFrame(checks)
display(results)

print("\nChecks passed:", int(results["Passed"].sum()), "/", len(results))
print("Checks requiring investigation:",
      results.loc[~results["Passed"], "Check"].tolist())
print("\nStep 28C completed. No artifacts were modified.")

,Check,Passed,Details
0,Holdout and label lengths,True,"holdout=19875, baseline=19875, Step23=19875"
1,Baseline labels match holdout,True,Compared in current row order
2,Step23 labels match holdout,True,Compared in current row order
3,Baseline prediction lengths,True,"probabilities=19875, predictions=19875"
4,Baseline threshold consistency,True,Saved threshold=0.162020
5,Baseline probabilities valid,True,"All probabilities should be finite and in [0, 1]"
6,Step23 prediction lengths,True,"probabilities=19875, predictions=19875"
7,Step23 threshold consistency,True,Saved threshold=0.162020
8,Step23 probabilities valid,True,"All probabilities should be finite and in [0, 1]"
9,Fairness holdout count,True,Saved n_holdout=19875; actual=19875



Checks passed: 18 / 18
Checks requiring investigation: []

Step 28C completed. No artifacts were modified.


**Step 29 — Final research decision and audit report**

In [37]:

# STEP 29 — Final model decision and research report

from pathlib import Path
import pandas as pd

D = Path("/content/drive/MyDrive/DiaVigil/data/processed")

comparison = pd.read_csv(D / "step27_model_comparison.csv")
bootstrap = pd.read_csv(D / "step27_bootstrap_summary.csv")

required = {
    "model", "PR_AUC", "ROC_AUC", "Brier",
    "Recall", "FNR", "FP", "FN"
}
assert required.issubset(comparison.columns), (
    f"Unexpected comparison schema: {comparison.columns.tolist()}"
)

baseline = comparison.loc[
    comparison["model"].eq("Original baseline")
].iloc[0]

candidate = comparison.loc[
    comparison["model"].eq("Step 23 model")
].iloc[0]

assert len(comparison.loc[
    comparison["model"].eq("Original baseline")
]) == 1
assert len(comparison.loc[
    comparison["model"].eq("Step 23 model")
]) == 1

report = f"""# DiaVigil — Final Research Audit and Model Decision

## 1. Audit status

The Step 28C cross-artifact consistency checks passed 18/18.
This verifies the tested artifact relationships, not clinical validity.

## 2. Holdout model comparison

| Metric | Original baseline | Step 23 model |
|---|---:|---:|
| PR-AUC | {baseline['PR_AUC']:.6f} | {candidate['PR_AUC']:.6f} |
| ROC-AUC | {baseline['ROC_AUC']:.6f} | {candidate['ROC_AUC']:.6f} |
| Brier score | {baseline['Brier']:.6f} | {candidate['Brier']:.6f} |
| Recall | {baseline['Recall']:.6f} | {candidate['Recall']:.6f} |
| False-negative rate | {baseline['FNR']:.6f} | {candidate['FNR']:.6f} |
| False positives | {int(baseline['FP'])} | {int(candidate['FP'])} |
| False negatives | {int(baseline['FN'])} | {int(candidate['FN'])} |

## 3. Interpretation

The Step 23 model has a different operating-point trade-off.
Compare discrimination, calibration, missed cases, false alarms,
and the assumed costs of each error before selecting a champion.

A lower false-negative rate alone does not establish overall
superiority. Statistical uncertainty and subgroup performance
must inform the final decision.

## 4. Research limitations

- The holdout results are from the available dataset, not an
  independent external healthcare system.
- Subgroup estimates with small positive-outcome counts are uncertain.
- Calibration does not establish clinical usefulness.
- Estimated resource savings require defensible operational and
  financial assumptions.
- Agent-generated care directives require clinical review and
  evaluation of factual faithfulness and safety.

## 5. Recommended next work

1. Finalize the model-selection rationale using the paired bootstrap.
2. Document the dataset, outcome definition, cohort construction,
   patient-level split strategy, and threshold-selection procedure.
3. Validate the workflow on a genuinely independent cohort if available.
4. Report subgroup uncertainty and the limits of fairness conclusions.
5. Evaluate agent faithfulness, unsafe recommendations, and
   human oversight before proposing clinical use.

## 6. Intended status

DiaVigil is a research prototype. These results do not establish
readiness for clinical deployment or prospective clinical benefit.
"""

out = D / "step29_final_research_decision_report.md"
out.write_text(report, encoding="utf-8")

print("Step 29 completed.")
print("Report saved:", out)
print("\n", report)


Step 29 completed.
Report saved: /content/drive/MyDrive/DiaVigil/data/processed/step29_final_research_decision_report.md

 # DiaVigil — Final Research Audit and Model Decision

## 1. Audit status

The Step 28C cross-artifact consistency checks passed 18/18.
This verifies the tested artifact relationships, not clinical validity.

## 2. Holdout model comparison

| Metric | Original baseline | Step 23 model |
|---|---:|---:|
| PR-AUC | 0.203595 | 0.201322 |
| ROC-AUC | 0.664123 | 0.657848 |
| Brier score | 0.096154 | 0.096093 |
| Recall | 0.291518 | 0.319643 |
| False-negative rate | 0.708482 | 0.680357 |
| False positives | 2089 | 2478 |
| False negatives | 1587 | 1524 |

## 3. Interpretation

The Step 23 model has a different operating-point trade-off.
Compare discrimination, calibration, missed cases, false alarms,
and the assumed costs of each error before selecting a champion.

A lower false-negative rate alone does not establish overall
superiority. Statistical uncertainty and subgr

**Step 30 — Freeze the research artifacts**

In [38]:

# STEP 30 — Freeze and fingerprint DiaVigil research artifacts

from pathlib import Path
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

ROOT = Path("/content/drive/MyDrive/DiaVigil")
DATA_DIR = ROOT / "data" / "processed"

assert DATA_DIR.exists(), f"Directory not found: {DATA_DIR}"

# Record important research artifacts without modifying them.
key_files = [
    "dev_cohort.parquet",
    "audit_test_cohort.parquet",
    "feature_engineering_pipeline.joblib",
    "model_calibration_artifacts.joblib",
    "fairness_audit_results.joblib",
    "simulation_artifacts.joblib",
    "agent_audit_demonstration.joblib",
    "step23_holdout_evaluation.joblib",
    "step24_holdout_fairness_calibration.joblib",
    "step25_baseline_comparison.csv",
    "step26_paired_cluster_bootstrap.csv",
    "step27_model_comparison.csv",
    "step27_bootstrap_summary.csv",
    "step27_research_audit_report.md",
    "step29_final_research_decision_report.md",
]

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    "project": "DiaVigil",
    "manifest_created_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "packages": {},
    "artifacts": [],
}

for package in [
    "numpy", "pandas", "scikit-learn",
    "catboost", "joblib", "pyarrow"
]:
    try:
        manifest["packages"][package] = version(package)
    except PackageNotFoundError:
        manifest["packages"][package] = "not installed"

for filename in key_files:
    path = DATA_DIR / filename

    entry = {
        "filename": filename,
        "exists": path.is_file(),
    }

    if path.is_file():
        entry["size_bytes"] = path.stat().st_size
        entry["sha256"] = sha256_file(path)

    manifest["artifacts"].append(entry)

out = DATA_DIR / "step30_reproducibility_manifest.json"
out.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

present = sum(item["exists"] for item in manifest["artifacts"])

print("STEP 30 COMPLETED")
print(f"Key artifacts present: {present}/{len(key_files)}")
print("Manifest:", out)
print("\nMissing artifacts:")

missing = [
    item["filename"]
    for item in manifest["artifacts"]
    if not item["exists"]
]

print(missing if missing else "None")
print("\nRecorded package versions:")
print(json.dumps(manifest["packages"], indent=2))


STEP 30 COMPLETED
Key artifacts present: 15/15
Manifest: /content/drive/MyDrive/DiaVigil/data/processed/step30_reproducibility_manifest.json

Missing artifacts:
None

Recorded package versions:
{
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "scikit-learn": "1.6.1",
  "catboost": "1.2.10",
  "joblib": "1.6.0",
  "pyarrow": "23.0.1"
}


**Locate the saved artifact**

In [39]:

from pathlib import Path

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/DiaVigil/data/processed"
)

artifact_path = ARTIFACT_DIR / "simulation_artifacts.joblib"

print("Expected path:", artifact_path)
print("File exists:", artifact_path.exists())

if not artifact_path.exists():
    print("\nSearching Google Drive for the artifact...")
    drive_root = Path("/content/drive/MyDrive")

    matches = list(
        drive_root.rglob("simulation_artifacts.joblib")
    )

    if matches:
        for path in matches:
            print("Found:", path)
    else:
        print("Artifact not found anywhere under MyDrive.")


Expected path: /content/drive/MyDrive/DiaVigil/data/processed/simulation_artifacts.joblib
File exists: True


**Inspect the simulation results**

In [40]:

import joblib

simulation_artifacts = joblib.load(artifact_path)

print("Available keys:", list(simulation_artifacts.keys()))

for key in [
    "allocation_summary",
    "capacity_frontier",
    "monte_carlo_bounds"
]:
    print(f"\n{'=' * 60}")
    print(f"ARTIFACT: {key}")
    print("=" * 60)

    if key not in simulation_artifacts:
        print("MISSING — this key is not in the saved artifact.")
        continue

    value = simulation_artifacts[key]

    print("Type:", type(value).__name__)

    if hasattr(value, "shape"):
        print("Shape:", value.shape)
        display(value)

    elif isinstance(value, dict):
        for subkey, subvalue in value.items():
            print(f"{subkey}: {subvalue}")

    else:
        print(value)


Available keys: ['allocation_summary', 'capacity_frontier', 'monte_carlo_bounds']

ARTIFACT: allocation_summary
Type: DataFrame
Shape: (3, 6)


,Nurse Hour Budget,Patients Reached,Hours Utilized,Exp. Readmissions Averted,True Readmissions Caught,Cohort Precision %
0,500,176,499.8,21.35,64,36.36
1,1500,465,1498.0,52.48,131,28.17
2,3000,879,2997.5,91.15,224,25.48



ARTIFACT: capacity_frontier
Type: DataFrame
Shape: (25, 4)


,Nurse_Hours,Patients_Contacted,Readmissions_Averted,marginal_yield_per_100h
0,200.0,76,9.907189,NaN
1,400.0,144,17.726743,3.909777
2,600.0,206,24.832801,3.553029
3,800.0,269,31.465505,3.316352
4,1000.0,326,37.759646,3.147071
5,1200.0,387,43.851166,3.045760
6,1400.0,436,49.645093,2.896964
7,1600.0,499,55.337504,2.846206
8,1800.0,556,60.827991,2.745243
9,2000.0,609,66.185159,2.678584



ARTIFACT: monte_carlo_bounds
Type: dict
mean_averted: 231.55218345901264
ci_lower: 228.0604660768503
ci_upper: 235.12225317463805


In [41]:

bounds = simulation_artifacts["monte_carlo_bounds"]

print("Monte Carlo bounds:")
for key, value in bounds.items():
    print(f"{key}: {value}")

print("\nComplete capacity frontier:")
display(simulation_artifacts["capacity_frontier"])


Monte Carlo bounds:
mean_averted: 231.55218345901264
ci_lower: 228.0604660768503
ci_upper: 235.12225317463805

Complete capacity frontier:


,Nurse_Hours,Patients_Contacted,Readmissions_Averted,marginal_yield_per_100h
0,200.0,76,9.907189,NaN
1,400.0,144,17.726743,3.909777
2,600.0,206,24.832801,3.553029
3,800.0,269,31.465505,3.316352
4,1000.0,326,37.759646,3.147071
5,1200.0,387,43.851166,3.045760
6,1400.0,436,49.645093,2.896964
7,1600.0,499,55.337504,2.846206
8,1800.0,556,60.827991,2.745243
9,2000.0,609,66.185159,2.678584


**Agent 1: Confirm the actual retrieval mechanism used in the implementation.**

In [42]:

import json
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "05_diavigil_agentic_triage_and_faithfulness_critic.ipynb"
)

assert notebook_path.exists(), f"Notebook not found: {notebook_path}"

nb = json.loads(notebook_path.read_text(encoding="utf-8"))

search_terms = [
    "duckdb", "sql", "select ", "read_parquet",
    "retriev", "encounter_id", "patient_nbr",
    "run_pipeline"
]

for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    matches = [
        term for term in search_terms
        if term in source.lower()
    ]

    if matches:
        print(f"\n--- Cell {i} | Matches: {matches} ---")
        print(source[:3500])



--- Cell 2 | Matches: ['read_parquet'] ---
from google.colab import drive
import os
import joblib
import pandas as pd
import numpy as np
import re
from sklearn.base import BaseEstimator, TransformerMixin

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/DiaVigil'
DATA_PROCESSED = os.path.join(BASE_DIR, 'data/processed')

# Install explainability and typing dependencies
!pip install -q shap pydantic
import shap
from pydantic import BaseModel, Field
from typing import List, Literal

# Re-declare custom transformer classes for pipeline deserialization
ELIXHAUSER_ICD9_RULES = {
    'elix_congestive_heart_failure': r'^(39891|40201|40211|40291|40401|40403|40411|40413|40491|40493|428)',
    'elix_cardiac_arrhythmias': r'^(42610|42611|42613|4262|4263|4264|42651|42652|42653|4266|4267|4268|4270|4272|4273|4276|4278|4279|7850|V450|V533)',
    'elix_valvular_disease': r'^(0932|394|395|396|397|424|7463|7464|7465|7466|V422|V433)',
    'elix_pulmonary_circulation': r'^(416|4179)',
  

**Agent 2: Confirm that the saved model, calibrator, and local TreeSHAP output are used together as described.**

In [43]:

agent_terms = [
    "pipeline.transform",
    "predict_proba",
    "TreeExplainer",
    "shap_values",
    "extract_patient_shap",
    "primary_risk_drivers",
    "protective_factors",
    "optimal_threshold"
]

for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    matches = [
        term for term in agent_terms
        if term.lower() in source.lower()
    ]

    if matches:
        print(f"\n--- Cell {i} | Matches: {matches} ---")
        print(source[:4500])



--- Cell 2 | Matches: ['optimal_threshold'] ---
from google.colab import drive
import os
import joblib
import pandas as pd
import numpy as np
import re
from sklearn.base import BaseEstimator, TransformerMixin

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/DiaVigil'
DATA_PROCESSED = os.path.join(BASE_DIR, 'data/processed')

# Install explainability and typing dependencies
!pip install -q shap pydantic
import shap
from pydantic import BaseModel, Field
from typing import List, Literal

# Re-declare custom transformer classes for pipeline deserialization
ELIXHAUSER_ICD9_RULES = {
    'elix_congestive_heart_failure': r'^(39891|40201|40211|40291|40401|40403|40411|40413|40491|40493|428)',
    'elix_cardiac_arrhythmias': r'^(42610|42611|42613|4262|4263|4264|42651|42652|42653|4266|4267|4268|4270|4272|4273|4276|4278|4279|7850|V450|V533)',
    'elix_valvular_disease': r'^(0932|394|395|396|397|424|7463|7464|7465|7466|V422|V433)',
    'elix_pulmonary_circulation': r'^(416|4179)

**Agent 3 and critic: Confirm which checks are actually implemented and whether a failed check triggers a retry or simply records rejection.**

In [44]:

import joblib

artifact_path = Path(
    "/content/drive/MyDrive/DiaVigil/data/processed/"
    "agent_audit_demonstration.joblib"
)

assert artifact_path.exists(), f"Artifact not found: {artifact_path}"

agent_artifacts = joblib.load(artifact_path)

print("Artifact keys:", list(agent_artifacts.keys()))

for name in ["sample_passed_brief", "sample_rejected_brief"]:
    brief = agent_artifacts.get(name)

    print(f"\n--- {name} ---")

    if not isinstance(brief, dict):
        print("Missing or unexpected format:", type(brief).__name__)
        continue

    print("Available fields:", list(brief.keys()))
    print("Risk tier:", brief.get("risk_tier"))
    print("Critic passed:", brief.get("critic_audit_passed"))
    print("Critic notes:", brief.get("critic_audit_notes"))
    print(
        "Number of care directives:",
        len(brief.get("synthesized_care_directives", []))
    )


Artifact keys: ['sample_passed_brief', 'sample_rejected_brief']

--- sample_passed_brief ---
Available fields: ['encounter_id', 'patient_nbr', 'readmission_risk', 'risk_tier', 'primary_risk_drivers', 'protective_factors', 'synthesized_care_directives', 'critic_audit_passed', 'critic_audit_notes']
Risk tier: High Risk - Enhanced Follow-up
Critic passed: True
Critic notes: AUDIT PASSED: Care directives ground directly into verified risk drivers (3 verified).
Number of care directives: 3

--- sample_rejected_brief ---
Available fields: ['encounter_id', 'patient_nbr', 'readmission_risk', 'risk_tier', 'primary_risk_drivers', 'protective_factors', 'synthesized_care_directives', 'critic_audit_passed', 'critic_audit_notes']
Risk tier: High Risk - Enhanced Follow-up
Critic passed: False
Critic notes: AUDIT FAILED (HALLUCINATION DETECTED): Directives cite clinical factors unsupported by local TreeSHAP attributions. Regeneration required.
Number of care directives: 2


In [45]:

critic_terms = [
    "critic", "critic_audit_passed", "critic_audit_notes",
    "simulate_hallucination", "retry", "regenerat",
    "passed", "rejected", "run_pipeline"
]

for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    matches = [
        term for term in critic_terms
        if term.lower() in source.lower()
    ]

    if matches:
        print(f"\n--- Cell {i} | Matches: {matches} ---")
        print(source[:5000])



--- Cell 6 | Matches: ['critic', 'critic_audit_passed', 'critic_audit_notes', 'passed'] ---
class FeatureImpact(BaseModel):
    feature: str
    attribution: float

class PatientTriageBrief(BaseModel):
    encounter_id: int
    patient_nbr: int
    readmission_risk: float = Field(ge=0.0, le=1.0)
    risk_tier: Literal["Standard Care", "High Risk - Enhanced Follow-up"]
    primary_risk_drivers: List[FeatureImpact]
    protective_factors: List[FeatureImpact]
    synthesized_care_directives: List[str]
    critic_audit_passed: bool = False
    critic_audit_notes: str = ""

print("Pydantic schema definitions compiled.")

--- Cell 7 | Matches: ['critic'] ---
**Cell 4: Agent Orchestration Engine with Deterministic Critic Gate**

Build the end-to-end multi-agent pipeline:

Agent 1 (Clinical Retriever): Safely pulls encounter parameters.

Agent 2 (Inference Core): Computes calibrated probability and local TreeSHAP drivers.

Agent 3 (Synthesis Engine): Synthesizes actionable care directives.

C

**inspect the manifest**

In [46]:

import json
from pathlib import Path

manifest_path = Path(
    "/content/drive/MyDrive/DiaVigil/data/processed/"
    "step30_reproducibility_manifest.json"
)

assert manifest_path.exists(), f"Manifest not found: {manifest_path}"

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

print(json.dumps(manifest, indent=2))


{
  "project": "DiaVigil",
  "manifest_created_utc": "2026-10-23T21:10:18.267009+00:00",
  "python_version": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "packages": {
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "scikit-learn": "1.6.1",
    "catboost": "1.2.10",
    "joblib": "1.6.0",
    "pyarrow": "23.0.1"
  },
  "artifacts": [
    {
      "filename": "dev_cohort.parquet",
      "exists": true,
      "size_bytes": 2192155,
      "sha256": "b3d6de81aebbed948887858f4593c0da9c6df2dce56f2744c02a0feb57cc8595"
    },
    {
      "filename": "audit_test_cohort.parquet",
      "exists": true,
      "size_bytes": 566805,
      "sha256": "534a4b127c74314d89e364cda5c88d4787fde5f3f54888b375d9b7ccd82a21ca"
    },
    {
      "filename": "feature_engineering_pipeline.joblib",
      "exists": true,
      "size_bytes": 2346,
      "sha256": "6dc062d08bb0e592ceaf97fb2869173d671fd11547ee8b2cd515c33c107eea69"
    },
    {
      "file

**Verify artifact integrity**

In [47]:

import hashlib
import json
from pathlib import Path

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/DiaVigil/data/processed"
)

MANIFEST_PATH = ARTIFACT_DIR / "step30_reproducibility_manifest.json"

with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

results = []

for item in manifest["artifacts"]:
    path = ARTIFACT_DIR / item["filename"]

    if not path.is_file():
        results.append({
            "filename": item["filename"],
            "exists_now": False,
            "size_matches": False,
            "sha256_matches": False,
            "status": "MISSING"
        })
        continue

    current_size = path.stat().st_size

    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    current_hash = digest.hexdigest()
    size_matches = current_size == item["size_bytes"]
    hash_matches = current_hash == item["sha256"]

    results.append({
        "filename": item["filename"],
        "exists_now": True,
        "size_matches": size_matches,
        "sha256_matches": hash_matches,
        "status": (
            "PASS" if size_matches and hash_matches
            else "MISMATCH"
        )
    })

import pandas as pd

verification = pd.DataFrame(results)
display(verification)

print("\nArtifact integrity summary")
print(verification["status"].value_counts().to_dict())
print("Total artifacts checked:", len(verification))
print("All artifacts verified:", (verification["status"] == "PASS").all())


,filename,exists_now,size_matches,sha256_matches,status
0,dev_cohort.parquet,True,True,True,PASS
1,audit_test_cohort.parquet,True,True,True,PASS
2,feature_engineering_pipeline.joblib,True,True,True,PASS
3,model_calibration_artifacts.joblib,True,True,True,PASS
4,fairness_audit_results.joblib,True,True,True,PASS
5,simulation_artifacts.joblib,True,True,True,PASS
6,agent_audit_demonstration.joblib,True,True,True,PASS
7,step23_holdout_evaluation.joblib,True,True,True,PASS
8,step24_holdout_fairness_calibration.joblib,True,True,True,PASS
9,step25_baseline_comparison.csv,True,True,True,PASS



Artifact integrity summary
{'PASS': 15}
Total artifacts checked: 15
All artifacts verified: True
